# Conformational Analysis I
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 23 · Monday, October 26, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Section 17.4 (review) and 17.8.1–17.8.3

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L23_Conformational_Analysis_I.ipynb)

> **New:** Parts 2–6 are **lecture notes**: key ideas, computed torsion profiles, chair conformations, worked examples and calculators. Read them before class or use them afterwards for review. The timed pair activity begins at Part 7.

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\", topic=\"\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n        self.topic = topic\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n\n\n    # ---------- diagnostic review (used by the L28 review session) ----------\ndef _topic_table(gb, guide=None, section=None):\n    \"\"\"Score by topic and say what to revisit. Attached to Gradebook below.\n\n    `section` limits the table to one section, so the mid-session diagnostic\n    does not count questions the student has not reached yet.\n    \"\"\"\n    rows = {}\n    for qid, it in gb.items.items():\n        if not it.topic:\n            continue\n        if section and it.section != section:\n            continue\n        got = gb.responses.get(qid)\n        r = rows.setdefault(it.topic, [0, 0, 0])\n        r[1] += 1\n        if got is not None:\n            r[2] += 1\n            if got == it.key:\n                r[0] += 1\n    if not rows:\n        print(\"No topic-tagged questions have been answered yet.\")\n        return {}\n    print(_rule(\"=\"))\n    print(\"  Where you stand, by lecture\")\n    print(_rule(\"=\"))\n    print(f\"  {'lecture':<9}{'answered':>10}{'correct':>9}{'':>4}\")\n    weak = []\n    for t in sorted(rows):\n        ok, total, ans = rows[t]\n        pct = (100.0 * ok / ans) if ans else 0.0\n        bar = \"#\" * int(round(pct / 10)) + \".\" * (10 - int(round(pct / 10)))\n        flag = \"\"\n        if ans == 0:\n            bar, flag = \"-\" * 10, \"  not attempted\"\n        elif pct < 67:\n            flag = \"  <- revisit\"\n            weak.append(t)\n        print(f\"  {t:<9}{ans:>5} of {total:<4}{ok:>5}   {bar}{flag}\")\n    print()\n    if weak:\n        print(\"  Before the exam, go back to:\")\n        for t in weak:\n            where = (guide or {}).get(t, \"\")\n            print(f\"    {t}  {where}\" if where else f\"    {t}\")\n    else:\n        done = sum(r[2] for r in rows.values())\n        total = sum(r[1] for r in rows.values())\n        if done < total:\n            print(f\"  Nothing below two thirds so far, with {total - done} \"\n                  f\"question(s) still unanswered.\")\n        else:\n            print(\"  No lecture is below two thirds. Review your missed \"\n                  \"questions above,\\n  then re-run the activities you found \"\n                  \"slowest rather than re-reading.\")\n    print(_rule(\"=\"))\n    return {t: dict(correct=v[0], answered=v[2], total=v[1]) for t, v in rows.items()}\n\n\nGradebook.topic_report = _topic_table\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"acetic acid (homologated)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"propanoic acid (homologated)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb

_cr = types.ModuleType("crystal"); _cr.__dict__["__name__"] = "crystal"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"PHS 312 \u2014 real-structure activities built on crystallographic coordinates.\n\nEverything here operates on a PDB entry embedded in the notebook itself.\nNo network, no install beyond NumPy and matplotlib. Optional 3D viewing via\npy3Dmol when it is available; otherwise matplotlib draws the structure.\n\nStructure: PDB 1ACJ, Harel et al., acetylcholinesterase with tacrine (THA)\nbound in the active-site gorge.\n\"\"\"\nimport base64, gzip, itertools, math\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom mpl_toolkits.mplot3d import Axes3D  # noqa: F401\n\nPRIMARY, SECOND, MUTED, OK = \"#0B6E8F\", \"#C2410C\", \"#6B7280\", \"#15803D\"\nPDB_B64 = \"...\"          # filled in by the notebook builder\n_cache = {}\n\n# covalent radii, enough to infer bonds from coordinates\n_COV = {\"C\": 0.76, \"N\": 0.71, \"O\": 0.66, \"S\": 1.05, \"H\": 0.31, \"P\": 1.07}\nVDW = {\"C\": 1.70, \"N\": 1.55, \"O\": 1.52, \"S\": 1.80, \"H\": 1.20, \"P\": 1.80}\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\ndef pdb_text():\n    if \"txt\" not in _cache:\n        _cache[\"txt\"] = gzip.decompress(base64.b64decode(PDB_B64)).decode()\n    return _cache[\"txt\"]\n\n\ndef _parse():\n    if \"parsed\" in _cache:\n        return _cache[\"parsed\"]\n    prot, lig = [], []\n    for l in pdb_text().splitlines():\n        if l.startswith((\"ATOM\", \"HETATM\")):\n            rec = dict(name=l[12:16].strip(), resn=l[17:20].strip(),\n                       chain=l[21].strip(), resi=l[22:26].strip(),\n                       x=float(l[30:38]), y=float(l[38:46]), z=float(l[46:54]),\n                       elem=(l[76:78].strip() or l[12:16].strip()[0]).upper())\n            (lig if l.startswith(\"HETATM\") else prot).append(rec)\n    _cache[\"parsed\"] = (prot, lig)\n    return prot, lig\n\n\ndef ligand(resn=\"THA\"):\n    _, lig = _parse()\n    if not isinstance(resn, str):\n        raise TypeError(\n            f\"ligand() wants a residue name such as 'THA', got {resn!r}. \"\n            \"If you meant a distance, pass it as cutoff=...\")\n    a = [r for r in lig if r[\"resn\"] == resn.upper()]\n    if not a:\n        have = sorted({r[\"resn\"] for r in lig})\n        raise ValueError(f\"no residue {resn!r} in this structure; available: {have}\")\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in a])\n    return a, X\n\n\ndef protein():\n    prot, _ = _parse()\n    X = np.array([[r[\"x\"], r[\"y\"], r[\"z\"]] for r in prot])\n    return prot, X\n\n\ndef bonds_of(atoms, X, tol=0.45):\n    \"\"\"Infer bonds from covalent radii. Returns (i, j, distance).\"\"\"\n    out = []\n    for i, j in itertools.combinations(range(len(X)), 2):\n        d = float(np.linalg.norm(X[i] - X[j]))\n        lim = _COV.get(atoms[i][\"elem\"], 0.77) + _COV.get(atoms[j][\"elem\"], 0.77) + tol\n        if d <= lim:\n            out.append((i, j, d))\n    return out\n\n\ndef angles_of(atoms, X, bonds):\n    adj = {}\n    for i, j, _ in bonds:\n        adj.setdefault(i, []).append(j); adj.setdefault(j, []).append(i)\n    out = []\n    for c, ns in adj.items():\n        for a, b in itertools.combinations(sorted(ns), 2):\n            v1, v2 = X[a] - X[c], X[b] - X[c]\n            cos = float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))\n            out.append((a, c, b, math.degrees(math.acos(max(-1, min(1, cos))))))\n    return out\n\n\n# ---------------------------------------------------------------- L22\ndef measure_geometry(term=\"bond lengths\", resn=\"THA\", show=True):\n    \"\"\"Measure what a force field parameterises, from the real crystal structure.\"\"\"\n    atoms, X = ligand(resn)\n    bl = bonds_of(atoms, X)\n    if term == \"bond lengths\":\n        vals = [d for _, _, d in bl]\n        labels = [f\"{atoms[i]['name']}-{atoms[j]['name']}\" for i, j, _ in bl]\n        unit, ideal, title = \"angstrom\", None, \"every bond in the ligand\"\n    else:\n        ang = angles_of(atoms, X, bl)\n        vals = [v for *_, v in ang]\n        labels = [f\"{atoms[a]['name']}-{atoms[c]['name']}-{atoms[b]['name']}\"\n                  for a, c, b, _ in ang]\n        unit, ideal, title = \"degrees\", (109.5, 120.0), \"every valence angle\"\n\n    if show:\n        f, ax = _fig(10.0, 3.8)\n        order = np.argsort(vals)\n        ax.bar(range(len(vals)), [vals[i] for i in order], color=PRIMARY)\n        ax.set_xticks(range(len(vals)))\n        ax.set_xticklabels([labels[i] for i in order], rotation=90, fontsize=7)\n        ax.set_ylabel(unit)\n        ax.set_ylim(min(vals) * 0.97, max(vals) * 1.02)\n        if ideal:\n            for v, lab in zip(ideal, (\"sp3  109.5\", \"sp2  120\")):\n                ax.axhline(v, color=SECOND, ls=\"--\", lw=1.4)\n                ax.text(len(vals) - .5, v, \"  \" + lab, va=\"center\",\n                        color=SECOND, fontsize=9)\n        ax.set_title(f\"{title}, measured in the crystal structure\")\n        plt.tight_layout(); plt.show()\n\n    v = np.array(vals)\n    if not show:\n        out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n                   hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n        return out\n    print(f\"{len(v)} measurements from the real structure\")\n    print(f\"  range  {v.min():.3f} to {v.max():.3f} {unit}\")\n    print(f\"  mean   {v.mean():.3f}   spread {v.std():.3f}\")\n    out = dict(term=term, n=len(v), lo=round(float(v.min()), 3),\n               hi=round(float(v.max()), 3), mean=round(float(v.mean()), 3))\n    if term == \"bond lengths\":\n        ar = [d for _, _, d in bl if 1.33 <= d <= 1.43]\n        sp3 = [d for _, _, d in bl if d > 1.48]\n        print(f\"  aromatic-like (1.33-1.43 A): {len(ar)}\")\n        print(f\"  single sp3-like (> 1.48 A) : {len(sp3)}\")\n        print(\"\\n  One force-field parameter cannot cover both. That is why a\")\n        print(\"  force field carries a different constant for each bond type.\")\n        out.update(aromatic=len(ar), sp3=len(sp3))\n    else:\n        n120 = int(((v > 112) & (v < 128)).sum()); n109 = int(((v > 103) & (v < 116)).sum())\n        print(f\"  near 120 deg (sp2): {n120}      near 109.5 deg (sp3): {n109}\")\n        print(\"\\n  The angles cluster around the two hybridisation ideals, which\")\n        print(\"  is exactly what the bending term is fitted to reproduce.\")\n        out.update(near_120=n120, near_109=n109)\n    return out\n\n\ndef contacts(cutoff=4.5, resn=\"THA\", show=True):\n    \"\"\"Real non-bonded contacts between the ligand and the protein.\"\"\"\n    atoms, L = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(L[:, None, :] - P[None, :, :], axis=2)\n    pairs = []\n    for i in range(len(L)):\n        for j in np.where(d[i] <= cutoff)[0]:\n            r = prot[int(j)]\n            s = (VDW.get(atoms[i][\"elem\"], 1.7) + VDW.get(r[\"elem\"], 1.7))\n            pairs.append((float(d[i, int(j)]), atoms[i][\"name\"], r[\"resn\"],\n                          r[\"resi\"], r[\"name\"], s))\n    pairs.sort()\n    if show:\n        gap = np.array([p[0] - p[5] for p in pairs])\n        col = [\"#DC2626\" if g < -0.4 else (OK if g < 0.5 else MUTED) for g in gap]\n        f, ax = _fig(9.4, 4.2)\n        ax.bar(range(len(gap)), gap, color=col, width=1.0)\n        ax.axhline(0, color=\"#1F2937\", lw=1.6)\n        ax.set_xlabel(f\"ligand-protein atom pairs within {cutoff} A, closest first\")\n        ax.set_ylabel(\"distance minus sum of vdW radii (A)\")\n        ax.set_title(\"zero means exactly at van der Waals contact;\\n\"\n                     \"a real clash sits well below zero, and there are none here\",\n                     fontsize=10)\n        nc = int((gap < -0.4).sum()); nt = int(((gap >= -0.4) & (gap < 0.5)).sum())\n        ax.legend(handles=[\n            plt.Line2D([], [], color=\"#DC2626\", lw=6, label=f\"clash ({nc})\"),\n            plt.Line2D([], [], color=OK, lw=6, label=f\"at contact ({nt})\"),\n            plt.Line2D([], [], color=MUTED, lw=6, label=\"further out\")],\n            fontsize=8, loc=\"lower right\")\n        plt.tight_layout(); plt.show()\n    tight = [p for p in pairs if p[0] < p[5]]\n    clash = [p for p in pairs if p[0] < p[5] - 0.4]\n    if show:\n        print(f\"{len(pairs)} ligand-protein atom pairs within {cutoff} A\")\n        print(f\"  {'dist':>6}  {'sum vdW':>8}  contact\")\n        for p in pairs[:10]:\n            mark = \"  at/inside vdW contact\" if p[0] < p[5] else \"\"\n            print(f\"  {p[0]:>6.2f}  {p[5]:>8.2f}  {p[1]:<4} ... \"\n                  f\"{p[2]}{p[3]} {p[4]:<4}{mark}\")\n        print(f\"\\n  at or just inside the vdW sum : {len(tight)}\")\n        print(f\"  genuinely clashing (>0.4 A in): {len(clash)}\")\n        print()\n        if not clash:\n            print(\"  Nothing clashes. That is what a real binding mode looks like:\")\n            print(\"  the ligand sits AT van der Waals contact with the protein,\")\n            print(\"  never inside it. A docking pose that scores well while pushing\")\n            print(\"  atoms closer than this is wrong, however good its number.\")\n    resid = sorted({(p[2], p[3]) for p in pairs})\n    return dict(cutoff=cutoff, pairs=len(pairs), at_or_inside_vdw=len(tight),\n                clashing=len(clash), residues=len(resid),\n                nearest=round(pairs[0][0], 2) if pairs else None)\n\n\ndef dihedral(p0, p1, p2, p3):\n    \"\"\"Standard IUPAC torsion angle in degrees.\"\"\"\n    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2\n    b1n = b1 / np.linalg.norm(b1)\n    v = b0 - (b0 @ b1n) * b1n\n    w = b2 - (b2 @ b1n) * b1n\n    x = v @ w\n    y = np.cross(b1n, v) @ w\n    return math.degrees(math.atan2(y, x))\n\n\nCHI1 = {\"ARG\": \"CG\", \"ASN\": \"CG\", \"ASP\": \"CG\", \"CYS\": \"SG\", \"GLN\": \"CG\",\n        \"GLU\": \"CG\", \"HIS\": \"CG\", \"ILE\": \"CG1\", \"LEU\": \"CG\", \"LYS\": \"CG\",\n        \"MET\": \"CG\", \"PHE\": \"CG\", \"PRO\": \"CG\", \"SER\": \"OG\", \"THR\": \"OG1\",\n        \"TRP\": \"CG\", \"TYR\": \"CG\", \"VAL\": \"CG1\"}\n\n\ndef sidechain_torsions(near_ligand=8.0, resn=\"THA\"):\n    \"\"\"Real chi1 torsions of residues lining the pocket.\"\"\"\n    prot, P = protein()\n    _, L = ligand(resn)\n    byres = {}\n    for i, r in enumerate(prot):\n        byres.setdefault((r[\"chain\"], r[\"resi\"], r[\"resn\"]), {})[r[\"name\"]] = P[i]\n    out = []\n    for (ch, ri, rn), at in byres.items():\n        g = CHI1.get(rn)\n        if not g or not all(k in at for k in (\"N\", \"CA\", \"CB\", g)):\n            continue\n        d = float(np.min(np.linalg.norm(L - at[\"CA\"], axis=1)))\n        if d > near_ligand:\n            continue\n        chi = dihedral(at[\"N\"], at[\"CA\"], at[\"CB\"], at[g])\n        out.append(dict(resn=rn, resi=ri, chain=ch, chi1=chi, dist=d))\n    out.sort(key=lambda r: r[\"dist\"])\n    return out\n\n\ndef rotamers(near_ligand=8.0, show=True):\n    \"\"\"Real side-chain torsions cluster at the three staggered minima.\"\"\"\n    rows = sidechain_torsions(near_ligand)\n    chi = np.array([r[\"chi1\"] for r in rows])\n    def bucket(a):\n        a = (a + 360) % 360\n        if 0 <= a < 120: return \"gauche+ (~60)\"\n        if 120 <= a < 240: return \"trans (~180)\"\n        return \"gauche- (~300)\"\n    buckets = [bucket(c) for c in chi]\n    if show:\n        f, ax = _fig(9.4, 4.0)\n        ax.hist(((chi + 360) % 360), bins=36, range=(0, 360), color=PRIMARY)\n        for v, lab in ((60, \"gauche+\"), (180, \"trans\"), (300, \"gauche-\")):\n            ax.axvline(v, color=SECOND, ls=\"--\", lw=1.5)\n            ax.text(v, ax.get_ylim()[1] * .92, \"  \" + lab, color=SECOND, fontsize=9)\n        ax.set_xlim(0, 360); ax.set_xticks(range(0, 361, 60))\n        ax.set_xlabel(\"chi-1 torsion angle (degrees)\")\n        ax.set_ylabel(\"side chains\")\n        ax.set_title(f\"{len(rows)} residues within {near_ligand} A of the ligand, \"\n                     \"measured in the crystal\")\n        plt.tight_layout(); plt.show()\n        print(f\"{'residue':<10}{'chi-1':>9}   rotamer                 dist to ligand\")\n        for r in rows[:12]:\n            print(f\"  {r['resn']}{r['resi']:<6}{r['chi1']:>9.1f}   \"\n                  f\"{bucket(r['chi1']):<22}  {r['dist']:.1f} A\")\n        from collections import Counter\n        c = Counter(buckets)\n        print(f\"\\n  {c}\")\n        print(\"\\n  Side chains do not sit at random angles. They occupy three\")\n        print(\"  staggered wells, which is the same local-minimum structure you\")\n        print(\"  see in a torsion profile - measured here in a real protein.\")\n    from collections import Counter\n    c = Counter(buckets)\n    return dict(n=len(rows), counts=dict(c),\n                near_stagger=int(sum(1 for a in chi\n                                     if min(abs(((a + 360) % 360) - v)\n                                            for v in (60, 180, 300)) < 40)))\n\n\ndef search_cost(n_rot=7, step=30, seconds_each=0.05, show=True):\n    \"\"\"Arithmetic, not simulation: how many structures a grid search needs.\"\"\"\n    per_bond = int(round(360 / step))\n    n = per_bond ** n_rot\n    secs = n * seconds_each\n    yrs = secs / 3600 / 24 / 365\n    if show:\n        print(f\"settings per bond        {per_bond}   ({step} degree steps)\")\n        print(f\"rotatable bonds          {n_rot}\")\n        print(f\"conformations to score   {n:,}\")\n        print(f\"at {seconds_each} s each            {secs/3600:,.1f} hours\", end=\"\")\n        print(f\"   =  {yrs:,.0f} years\" if yrs >= 1 else \"\")\n        if yrs >= 1:\n            print(\"\\n  This is why no docking program does a systematic search.\")\n    return dict(conformations=n, per_bond=per_bond, hours=secs / 3600,\n                years=yrs)\n\n\n# ---------------------------------------------------------------- L24\ndef rmsd(A, B):\n    return float(np.sqrt(((A - B) ** 2).sum(axis=1).mean()))\n\n\ndef move_ligand(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\"):\n    \"\"\"Rigid-body move of the real ligand. Returns the moved coordinates.\"\"\"\n    _, X = ligand(resn)\n    c = X.mean(axis=0)\n    t = math.radians(rot_deg)\n    ax = {\"x\": np.array([1., 0, 0]), \"y\": np.array([0, 1., 0]),\n          \"z\": np.array([0, 0, 1.])}[axis]\n    K = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]])\n    R = np.eye(3) + math.sin(t) * K + (1 - math.cos(t)) * (K @ K)\n    return (X - c) @ R.T + c + np.array([dx, dy, dz])\n\n\ndef pose_rmsd(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, resn=\"THA\", show=True):\n    \"\"\"Move the real crystal pose and measure the real RMSD you created.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, resn=resn)\n    r = rmsd(X, Y)\n    per = np.linalg.norm(X - Y, axis=1)\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.2))\n        a1.bar(range(len(per)), per, color=PRIMARY)\n        a1.set_xticks(range(len(per)))\n        a1.set_xticklabels([a[\"name\"] for a in atoms], rotation=90, fontsize=7)\n        a1.set_ylabel(\"displacement (A)\")\n        a1.axhline(r, color=SECOND, ls=\"--\", lw=1.6, label=f\"RMSD = {r:.2f} A\")\n        a1.legend(fontsize=9)\n        a1.set_title(\"how far each atom moved\")\n        a2.scatter(X[:, 0], X[:, 1], s=60, color=MUTED, label=\"crystal pose\")\n        a2.scatter(Y[:, 0], Y[:, 1], s=60, color=SECOND, label=\"your pose\")\n        for i in range(len(X)):\n            a2.plot([X[i, 0], Y[i, 0]], [X[i, 1], Y[i, 1]], color=\"#AAB\", lw=.8)\n        a2.set_aspect(\"equal\"); a2.legend(fontsize=9)\n        a2.set_xlabel(\"x (A)\"); a2.set_ylabel(\"y (A)\")\n        a2.set_title(\"looking down z\")\n        for ax_ in (a1, a2):\n            ax_.grid(alpha=.25)\n            for s in (\"top\", \"right\"): ax_.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n    verdict = (\"counts as the correct pose (under 2 A)\" if r < 2.0\n               else \"would be scored as a wrong pose (2 A or more)\")\n    print(f\"RMSD to the crystal pose : {r:.2f} A   -> {verdict}\")\n    print(f\"furthest atom moved      : {per.max():.2f} A\")\n    print(f\"closest atom moved       : {per.min():.2f} A\")\n    return dict(rmsd=round(r, 2), max_move=round(float(per.max()), 2),\n                correct_pose=bool(r < 2.0))\n\n\ndef pharmacophore(resn=\"THA\", show=True):\n    \"\"\"The real 3-point geometry of the bound ligand.\"\"\"\n    atoms, X = ligand(resn)\n    idx = {a[\"name\"]: i for i, a in enumerate(atoms)}\n    ring = [idx[n] for n in (\"C1\", \"C2\", \"C3\", \"C4\", \"C5\", \"C6\") if n in idx]\n    pts = {}\n    if ring: pts[\"aromatic centroid\"] = X[ring].mean(axis=0)\n    for nm, lab in ((\"N15\", \"amine N (donor)\"), (\"N7\", \"ring N-H+ (donor)\")):\n        if nm in idx: pts[lab] = X[idx[nm]]\n    names = list(pts)\n    print(\"Three-point pharmacophore, measured in the crystal structure:\")\n    out = {}\n    for a, b in itertools.combinations(names, 2):\n        d = float(np.linalg.norm(pts[a] - pts[b]))\n        out[f\"{a} - {b}\"] = round(d, 2)\n        print(f\"   {a:<22} to {b:<22} {d:5.2f} A\")\n    if show and len(names) == 3:\n        f, ax = _fig(6.4, 5.2)\n        P2 = np.array([pts[n] for n in names])\n        ax.scatter(P2[:, 0], P2[:, 1], s=180,\n                   color=[PRIMARY, SECOND, OK][:len(names)], zorder=3)\n        for i, n in enumerate(names):\n            ax.annotate(n, (P2[i, 0], P2[i, 1]), fontsize=9,\n                        xytext=(8, 6), textcoords=\"offset points\")\n        for i, j in itertools.combinations(range(len(names)), 2):\n            ax.plot(P2[[i, j], 0], P2[[i, j], 1], color=MUTED, lw=1.4)\n            m = P2[[i, j]].mean(axis=0)\n            ax.text(m[0], m[1], f\"{np.linalg.norm(P2[i]-P2[j]):.2f} A\",\n                    fontsize=8, color=MUTED)\n        ax.set_aspect(\"equal\"); ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.margins(0.28)\n        ax.set_title(\"the pharmacophore is this triangle, not the skeleton\",\n                     pad=16)\n        plt.tight_layout(); plt.show()\n    return out\n\n\n# ---------------------------------------------------------------- L25 / L26\ndef score_pose(Y, resn=\"THA\", cutoff=6.0):\n    \"\"\"A deliberately simple score over REAL coordinates.\n\n    contact reward - clash penalty - desolvation-ish term. It is transparent\n    on purpose: every piece is printed, so the point is not that this is a\n    good scoring function but that you can see exactly what it rewards.\n    \"\"\"\n    atoms, _ = ligand(resn)\n    prot, P = protein()\n    d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n    ser = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[:, None]\n    spr = np.array([VDW.get(r[\"elem\"], 1.7) for r in prot])[None, :]\n    s = ser + spr\n    near = d <= cutoff\n    gap = d - s\n    contact = float(np.sum(np.exp(-((gap[near]) ** 2) / 0.8)))\n    overlap = np.clip(-gap, 0, None)\n    clash = float(np.sum(overlap[near] ** 2) * 20)\n    polar_l = np.array([a[\"elem\"] in (\"N\", \"O\") for a in atoms])[:, None]\n    polar_p = np.array([r[\"elem\"] in (\"N\", \"O\") for r in prot])[None, :]\n    hb_mask = near & polar_l & polar_p & (d > 2.4) & (d < 3.5)\n    hbond = float(hb_mask.sum()) * 2.5\n    total = -(contact + hbond) + clash\n    return dict(score=round(total, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int((overlap > 0.4).any(axis=1).sum()))\n\n\ndef dock_pose(dx=0.0, dy=0.0, dz=0.0, rot_deg=0.0, axis=\"z\", resn=\"THA\",\n              show=True):\n    \"\"\"Move the real ligand in the real gorge and score it.\"\"\"\n    atoms, X = ligand(resn)\n    Y = move_ligand(dx, dy, dz, rot_deg, axis, resn)\n    sc = score_pose(Y, resn)\n    r = rmsd(X, Y)\n    if show:\n        prot, P = protein()\n        d = np.linalg.norm(Y[:, None, :] - P[None, :, :], axis=2)\n        nearby = np.where(d.min(axis=0) <= 8.0)[0]\n        f, ax = _fig(7.2, 6.0)\n        ax.scatter(P[nearby, 0], P[nearby, 1], s=14, color=\"#CBD5E1\",\n                   label=\"gorge atoms within 8 A\")\n        ax.scatter(X[:, 0], X[:, 1], s=70, color=MUTED, label=\"crystal pose\")\n        bad = np.zeros(len(Y), bool)\n        ov = np.clip((np.array([VDW.get(a[\"elem\"],1.7) for a in atoms])[:,None]\n                      + np.array([VDW.get(p[\"elem\"],1.7) for p in prot])[None,:]) - d, 0, None)\n        bad = (ov > 0.4).any(axis=1)\n        ax.scatter(Y[~bad, 0], Y[~bad, 1], s=90, color=SECOND, label=\"your pose\")\n        if bad.any():\n            ax.scatter(Y[bad, 0], Y[bad, 1], s=130, color=\"#DC2626\", marker=\"X\",\n                       label=\"clashing atom\")\n        ax.set_aspect(\"equal\"); ax.legend(fontsize=8, loc=\"best\")\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"score {sc['score']:+.1f}    RMSD to crystal {r:.2f} A\")\n        plt.tight_layout(); plt.show()\n    print(f\"score            {sc['score']:+8.2f}   (more negative is better)\")\n    print(f\"  contact        {sc['contact']:8.2f}\")\n    print(f\"  hydrogen bonds {sc['hbond']:8.2f}\")\n    print(f\"  clash penalty  {sc['clash']:8.2f}   from {sc['clashing_atoms']} atoms\")\n    print(f\"RMSD to crystal  {r:8.2f} A\")\n    return dict(**sc, rmsd=round(r, 2))\n\n\ndef score_vs_truth(n=220, spread=4.0, seed=7, resn=\"THA\", show=True):\n    \"\"\"Generate real perturbed poses; plot score against RMSD to the crystal.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        a = rng.uniform(-180, 180)\n        ax_ = \"xyz\"[rng.integers(0, 3)]\n        Y = move_ligand(*t, rot_deg=a, axis=ax_, resn=resn)\n        sc = score_pose(Y, resn)\n        rows.append((rmsd(X, Y), sc[\"score\"], sc[\"clashing_atoms\"]))\n    R = np.array([r[0] for r in rows]); S = np.array([r[1] for r in rows])\n    good_rmsd = R < 2.0\n    best = int(np.argmin(S))\n    if show:\n        f, ax = _fig(9.0, 5.0)\n        ax.scatter(R[~good_rmsd], S[~good_rmsd], s=26, color=MUTED,\n                   label=\"wrong pose (RMSD >= 2 A)\", alpha=.75)\n        ax.scatter(R[good_rmsd], S[good_rmsd], s=40, color=PRIMARY,\n                   label=\"correct pose (RMSD < 2 A)\")\n        ax.scatter([R[best]], [S[best]], s=150, marker=\"*\", color=SECOND,\n                   zorder=5, label=\"best-scoring pose\")\n        lo = float(S.min()); hi = float(np.percentile(S, 55))\n        pad = max(10.0, 0.10 * (hi - lo)); off = int((S > hi + pad).sum())\n        ax.set_ylim(lo - pad, hi + pad)\n        if off:\n            ax.text(0.99, 0.03, f\"{off} heavily clashing poses lie above this axis\",\n                    transform=ax.transAxes, ha=\"right\", va=\"bottom\",\n                    fontsize=8, color=MUTED)\n        ax.axvline(2.0, color=\"#B45309\", ls=\":\", lw=1.4)\n        ax.set_xlabel(\"RMSD to the crystal pose (A)\")\n        ax.set_ylabel(\"score (more negative = better)\")\n        ax.legend(fontsize=8)\n        ax.set_title(f\"{n} real poses of tacrine in the real gorge\")\n        plt.tight_layout(); plt.show()\n    picked_right = bool(R[best] < 2.0)\n    print(f\"{n} poses generated from the real crystal coordinates\")\n    print(f\"  poses within 2 A of the crystal : {int(good_rmsd.sum())}\")\n    print(f\"  best-scoring pose has RMSD      : {R[best]:.2f} A\")\n    print(f\"  did the score pick a correct pose? {'yes' if picked_right else 'NO'}\")\n    return dict(n=n, correct_available=int(good_rmsd.sum()),\n                best_rmsd=round(float(R[best]), 2),\n                best_score=round(float(S[best]), 2),\n                score_picked_correct=picked_right)\n\n\n# ---------------------------------------------------------------- L27\ndef empty_space(probe=1.6, grid=1.0, around=9.0, resn=\"THA\", show=True):\n    \"\"\"Find unfilled volume in the real gorge next to the bound ligand.\n\n    Only gorge atoms are considered and the grid is processed in chunks, so a\n    fine grid stays within memory.\n    \"\"\"\n    if not isinstance(resn, str):\n        raise TypeError(\"resn must be a residue name such as 'THA'; \"\n                        \"did you mean empty_space(probe=%r)?\" % resn)\n    if grid < 0.3:\n        raise ValueError(\"grid finer than 0.3 A is not useful here and is slow\")\n    atoms, X = ligand(resn)\n    prot, Pall = protein()\n    c = X.mean(axis=0)\n    # prune to atoms that could possibly matter\n    keep_p = np.linalg.norm(Pall - c, axis=1) <= (around + probe + 2.5)\n    P = Pall[keep_p]\n    rp = np.array([VDW.get(r[\"elem\"], 1.7) for r in np.array(prot)[keep_p]])[None, :]\n    rl = np.array([VDW.get(a[\"elem\"], 1.7) for a in atoms])[None, :]\n    rng_ = np.arange(-around, around + 1e-9, grid)\n    G = np.array([[x, y, z] for x in rng_ for y in rng_ for z in rng_]) + c\n    G = G[np.linalg.norm(G - c, axis=1) <= around]\n    free = np.empty(len(G), bool)\n    dlmin = np.empty(len(G))\n    notlig = np.empty(len(G), bool)\n    step = max(1, int(4e6 // max(len(P), 1)))\n    for i in range(0, len(G), step):\n        g = G[i:i + step]\n        dp = np.linalg.norm(g[:, None, :] - P[None, :, :], axis=2)\n        free[i:i + step] = (dp > (rp + probe)).all(axis=1)\n        dl = np.linalg.norm(g[:, None, :] - X[None, :, :], axis=2)\n        notlig[i:i + step] = (dl > (rl + probe)).all(axis=1)\n        dlmin[i:i + step] = dl.min(axis=1)\n    pocket = free & (dlmin < 8.0)\n    unfilled = pocket & notlig\n    if show:\n        zc = float(X[:, 2].mean()); slab = 2.0\n        sl = np.abs(G[:, 2] - zc) <= slab\n        lig = np.abs(X[:, 2] - zc) <= slab\n        f, ax = _fig(7.0, 6.0)\n        ax.scatter(G[pocket & ~notlig & sl, 0], G[pocket & ~notlig & sl, 1],\n                   s=9, color=\"#CBD5E1\", label=\"taken up by the ligand\")\n        ax.scatter(G[unfilled & sl, 0], G[unfilled & sl, 1], s=12, color=OK,\n                   label=f\"free, probe {probe} A\")\n        ax.scatter(X[lig, 0], X[lig, 1], s=80, color=SECOND, label=\"tacrine\")\n        ax.set_aspect(\"equal\"); ax.legend(fontsize=8, loc=\"upper right\")\n        ax.set_xlabel(\"x (A)\"); ax.set_ylabel(\"y (A)\")\n        ax.set_title(f\"a {2*slab:.0f} A slice through the gorge, not the whole \"\n                     f\"volume\\n{int((unfilled & sl).sum())} of the \"\n                     f\"{int(unfilled.sum())} free points lie in this slice\",\n                     fontsize=10)\n        plt.tight_layout(); plt.show()\n    vol = float(unfilled.sum()) * grid ** 3\n    print(f\"probe radius             {probe} A\")\n    print(f\"grid points sampled      {len(G):,} at {grid} A spacing\")\n    print(f\"free and near the ligand {unfilled.sum():>8,}  -> about {vol:,.0f} A^3\")\n    print(\"\\n  That volume is what a de novo program would try to fill, and why\")\n    print(\"  a first design should NOT fill all of it.\")\n    return dict(probe=probe, grid=grid, grid_points=len(G),\n                unfilled=int(unfilled.sum()), volume_A3=round(vol, 1))\n\n\ndef score_ranking(n=220, spread=2.0, seed=7, top=10, resn=\"THA\"):\n    \"\"\"Rank real poses by score and show the RMSD of each. The question is not\n    only whether the top pose is right but how well the score SEPARATES.\"\"\"\n    rng = np.random.default_rng(seed)\n    atoms, X = ligand(resn)\n    rows = []\n    for _ in range(n):\n        t = rng.normal(0, spread / 2, 3)\n        Y = move_ligand(*t, rot_deg=rng.uniform(-180, 180),\n                        axis=\"xyz\"[rng.integers(0, 3)], resn=resn)\n        rows.append((rmsd(X, Y), score_pose(Y, resn)[\"score\"]))\n    rows.sort(key=lambda r: r[1])\n    print(f\"top {top} of {n} poses, ranked by score\")\n    print(f\"  {'rank':>4}  {'score':>9}  {'RMSD':>6}   verdict\")\n    for i, (r, sc) in enumerate(rows[:top], 1):\n        print(f\"  {i:>4}  {sc:>9.2f}  {r:>5.2f} A   \"\n              f\"{'correct' if r < 2 else 'WRONG POSE'}\")\n    wrong = sum(1 for r, _ in rows[:top] if r >= 2)\n    print(f\"\\n  {wrong} of the top {top} scoring poses are the wrong pose.\")\n    return dict(top=top, wrong_in_top=wrong,\n                rows=[(round(r, 2), round(s, 2)) for r, s in rows[:top]])\n\n\n_RING = {\"TRP\": {\"CD2\", \"CE2\", \"CE3\", \"CZ2\", \"CZ3\", \"CH2\"},\n         \"PHE\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"TYR\": {\"CG\", \"CD1\", \"CD2\", \"CE1\", \"CE2\", \"CZ\"},\n         \"HIS\": {\"CG\", \"ND1\", \"CD2\", \"CE1\", \"NE2\"}}\n\n\ndef stacking(near=6.0, resn=\"THA\", show=True):\n    \"\"\"Is the ligand sandwiched? Measure each nearby aromatic ring's position\n    relative to the plane of the ligand itself.\"\"\"\n    atoms, X = ligand(resn)\n    prot, P = protein()\n    cen = X.mean(axis=0)\n    nrm = np.linalg.svd(X - cen)[2][2]\n    seen, rows = set(), []\n    for j, r in enumerate(prot):\n        kk = (r[\"resn\"], r[\"resi\"])\n        if r[\"resn\"] not in _RING or kk in seen:\n            continue\n        idx = [k for k, q in enumerate(prot)\n               if q[\"resn\"] == r[\"resn\"] and q[\"resi\"] == r[\"resi\"]\n               and q[\"name\"] in _RING[r[\"resn\"]]]\n        if len(idx) < 5:\n            continue\n        Q = P[idx]\n        qc = Q.mean(axis=0)\n        if np.linalg.norm(qc - cen) > near + 3.0:\n            continue\n        seen.add(kk)\n        qn = np.linalg.svd(Q - qc)[2][2]\n        signed = float(np.dot(qc - cen, nrm))\n        ang = float(np.degrees(np.arccos(min(1.0, abs(float(np.dot(nrm, qn)))))))\n        rows.append((f\"{r['resn']}{r['resi']}\", signed,\n                     float(np.linalg.norm(qc - cen)), ang))\n    rows.sort(key=lambda t: t[2])\n    rows = [t for t in rows if t[2] <= near + 1.0]\n    if show:\n        f, ax = _fig(7.6, 4.4)\n        for nm, sg, dd, an in rows:\n            st = an < 30\n            ax.barh(nm, sg, height=.6, color=(PRIMARY if sg > 0 else SECOND),\n                    alpha=(1.0 if st else 0.35),\n                    hatch=(None if st else \"///\"), edgecolor=\"white\")\n            ax.text(sg + (0.2 if sg > 0 else -0.2), nm,\n                    f\"{an:.0f} deg\" + (\"\" if st else \"  edge-on\"),\n                    va=\"center\", ha=(\"left\" if sg > 0 else \"right\"),\n                    fontsize=8, color=(\"#1F2937\" if st else MUTED))\n        ax.set_xlim(min(r[1] for r in rows) - 3.0, max(r[1] for r in rows) + 3.0)\n        ax.axvline(0, color=\"#1F2937\", lw=1.4)\n        ax.set_xlabel(\"signed distance from the ligand's plane (A)\")\n        ax.set_title(\"opposite sides of zero = sandwiched;\\nsolid bar = near-parallel = genuinely stacked\", fontsize=10)\n        plt.tight_layout(); plt.show()\n    print(\"aromatic rings near the ligand, measured against the ligand's own plane\")\n    print(f\"  {'residue':<9}{'side':>6}{'perp (A)':>10}{'centroid (A)':>14}\"\n          f\"{'planes (deg)':>14}\")\n    for nm, sg, dd, an in rows:\n        print(f\"  {nm:<9}{('above' if sg > 0 else 'below'):>6}{sg:>10.2f}\"\n              f\"{dd:>14.2f}{an:>14.1f}\")\n    above = [r for r in rows if r[1] > 0]\n    below = [r for r in rows if r[1] < 0]\n    print(f\"\\n  {len(above)} above the plane, {len(below)} below.\")\n    if above and below:\n        print(\"  Rings on both sides: the ligand is sandwiched.\")\n    print(\"  A plane angle under about 30 degrees means the rings are \"\n          \"roughly parallel,\\n  which is face-to-face stacking rather than \"\n          \"edge-on contact.\")\n    return dict(rows=[(n, round(s, 2), round(d, 2), round(a, 1))\n                      for n, s, d, a in rows],\n                above=len(above), below=len(below),\n                sandwiched=bool(above and below))\n", "crystal.py", "exec"), _cr.__dict__)
sys.modules["crystal"] = _cr
crystal = _cr

_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate.\"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.4))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    axes[0].set_ylabel(\"energy (kcal/mol)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[(Ev + Ec).argmin()]),\n                vdw_min_E=float((Ev + Ec).min()))\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    a2.hist(visited, bins=72, color=PRIMARY, alpha=.85)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\"))\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = [min(y.min(), pred.min(), loo.min()) - .4,\n               max(y.max(), pred.max(), loo.max()) + .4]\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1))\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    cost = {\"Molecular mechanics\": 1, \"Semi-empirical quantum mechanics\": 1e3,\n            \"Ab initio quantum mechanics\": 1e6}\n    f, ax = _fig(8.6, 2.9)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    ax.set_xscale(\"log\"); ax.set_xlabel(\"relative cost (log scale)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons)\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, excluded.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, excluded.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\")\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(9.6, 5.0)\n        ax.scatter([r[\"freq\"] for r in tab], [r[\"not_worse\"] * 100 for r in tab],\n                   s=70, color=PRIMARY, zorder=3)\n        for r in tab:\n            if r[\"freq\"] > 600 or r[\"not_worse\"] > 0.88:\n                ax.annotate(r[\"name\"], (r[\"freq\"], r[\"not_worse\"] * 100),\n                            fontsize=8, xytext=(5, 4), textcoords=\"offset points\")\n        ax.set_xscale(\"log\")\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout(); plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "A minimization run on one starting geometry returns a conformation. What has it established?",
         {"A": "That this is the lowest-energy shape the molecule can take", "B": "That every small change from this shape raises the energy", "C": "That this is the shape the molecule takes in the binding site", "D": "That no other conformation of the molecule is accessible"},
         "B", {"A": "That is the global minimum, and one downhill run cannot establish it.", "C": "The bound conformation depends on the target, which is not modelled here.", "D": "Other conformations remain accessible; they were simply not visited.", "_correct": "A minimum is defined locally: every small displacement raises the energy, and nothing more is proven."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "Why does a systematic search over rotatable bonds become impossible for a flexible drug?",
         {"A": "Because each additional bond adds its increments to the total", "B": "Because the force field loses accuracy as bond count increases", "C": "Because each additional bond multiplies the number of structures", "D": "Because rotatable bonds interact and cannot be scanned at all"},
         "C", {"A": "Addition would keep the problem small; the growth is multiplicative, not additive.", "D": "They can be scanned; the obstacle is the number of combinations, not feasibility.", "B": "Force-field accuracy does not depend on how many bonds you choose to scan.", "_correct": "Every bond must be scanned at every setting of the others, so the count is increments raised to the power of the number of bonds."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "In molecular dynamics, what is the purpose of raising the simulated temperature?",
         {"A": "To supply kinetic energy so barriers between minima are crossed", "B": "To lower the barriers themselves so less energy is required", "C": "To remove the non-bonded terms that hold the molecule folded", "D": "To speed the calculation by taking fewer integration steps"},
         "A", {"B": "Barrier heights are set by the force field and do not change with temperature.", "C": "Non-bonded terms stay in force throughout the simulation.", "D": "Higher temperature usually demands smaller steps, not fewer.", "_correct": "Temperature sets the kinetic energy available, and only a molecule with enough of it can climb out of a well and explore other minima."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "Starting at 100 degrees, minimization stopped at 60 degrees rather than at the global minimum. Why?",
         {"A": "A barrier lies between 100 degrees and the deeper 180 degree well", "B": "The 60 degree well is lower in energy than the 180 degree well", "C": "The algorithm found the global minimum but reported it as 60", "D": "The force field has no parameters for angles beyond 90 degrees"},
         "A", {"C": "The algorithm reported where it stopped; 60 degrees is a genuine local minimum.", "D": "The profile is calculated across the full rotation, so no range is missing.", "B": "The plot shows the 180 degree anti well is the deeper of the two.", "_correct": "Downhill minimization cannot climb, so a barrier between the start and the deeper well leaves it stranded in the nearer one."},
         bloom="Analyze", silo="6", section="concept"),
    Item("C2", "You raised the gauche strain until the wells near 60 degrees disappeared. What happens to minimization from 100 degrees now?",
         {"A": "It stops at 60 degrees, because that minimum is always present", "B": "It stops at 100 degrees, because there is nowhere left to go", "C": "It reaches 180 degrees, because the slope now runs all the way", "D": "It fails, because a profile with one minimum cannot be searched"},
         "C", {"A": "Raising the strain removed those wells, so there is no longer a minimum at 60 degrees.", "D": "A single-minimum surface is the easiest case there is for minimization.", "B": "A point on a slope is not a minimum, so the algorithm keeps descending.", "_correct": "With the intervening wells gone the surface slopes continuously toward 180 degrees, so a downhill run now reaches the global minimum."},
         bloom="Analyze", silo="6", section="concept"),
    Item("C3", "At 30 degree increments, 14 rotatable bonds gave a number beyond any practical computation. What does this establish about stepwise searching?",
         {"A": "It is unusable for flexible molecules because cost grows as a power", "B": "It is unusable for flexible molecules because the force field fails", "C": "It is usable if the computer available is sufficiently powerful", "D": "It is usable because most of those structures are chemically alike"},
         "A", {"B": "The force field is untroubled; the obstacle is purely the number of structures.", "C": "The count exceeds any foreseeable hardware, so more power does not rescue it.", "D": "Redundancy exists but does not change growth that is exponential in bond count.", "_correct": "Cost is increments raised to the power of the bond count, so each extra bond multiplies the work and flexible molecules are out of reach."},
         bloom="Analyze", silo="6", section="concept"),
    Item("C4", "Coarsening from 15 to 120 degree increments cut the work enormously. What is the cost of that choice?",
         {"A": "The force field can no longer calculate the steric energy at all", "B": "Narrow energy minima can fall between grid points and be missed", "C": "The global minimum is found but its energy is reported wrongly", "D": "Only cyclic molecules can be searched once the grid is coarse"},
         "B", {"C": "If the minimum is sampled its energy is right; the risk is not sampling it.", "D": "Grid spacing has nothing to do with whether a molecule is cyclic.", "A": "Energy is still calculated normally at each point that is sampled.", "_correct": "A coarse grid samples fewer dihedral values, so a minimum lying between two sampled angles is never evaluated."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C5", "At 8 angstrom your output measures 15 side chains, and every one lies within 40 degrees of a staggered value. What does that clustering establish?",
         {"A": "Binding-site side chains are unusually strained, as the clustering shows", "B": "Refinement restrained the torsions to ideal values, so this is circular", "C": "Fifteen residues is far too small a sample to show any real preference", "D": "Side chains occupy a few low-energy rotamers rather than arbitrary torsions"},
         "D", {"B": "Refinement restrains bond lengths and angles tightly but lets chi torsions follow the density; rotamer preference is an observation, not an imposed constraint.", "C": "Fifteen of fifteen falling in three narrow bands is a strong result, not a marginal one.", "A": "Clustering at staggered values is the opposite of strain; it shows the side chains are relaxed.", "_correct": "Rotamer libraries exist because this is what side chains do: eclipsed torsions are high in energy, so observed chi values cluster near staggered minima. Docking programs exploit exactly this by sampling rotamers instead of continuous torsions."},
         bloom="Analyze", silo="6", section="concept"),
    Item("C6", "Your output assigns Trp84 a chi-1 near -53 degrees and Phe330 near 157 degrees. Both are aromatic residues that stack against the ligand. What follows for a docking calculation that keeps the protein rigid?",
         {"A": "Yes regardless, because rigid docking still samples the side chains", "B": "Only if it starts from these rotamers, since it cannot find them itself", "C": "It fails outright, since rigid docking reproduces no crystal pose at all", "D": "Rotamers are irrelevant here, since docking only moves the ligand"},
         "B", {"C": "Rigid docking reproduces many crystal poses successfully when the input rotamers are the bound ones.", "D": "The side chains form the walls of the site, so their rotamers define the shape being docked into.", "A": "Rigid-receptor docking does not sample side chains; that is what rigid means.", "_correct": "Rigid-receptor docking inherits whatever rotamers the input structure has. Docking into a structure solved with a different ligand, where these side chains sit differently, can miss the correct pose entirely. This is the induced-fit problem, and it begins with the numbers you just measured."},
         bloom="Evaluate", silo="6", section="concept"),
    Item("C7", "Set rotatable_bonds to 12, keeping 30 degree steps: the output reports about 8.9 trillion conformations and some 14,000 years of computing. Halving the step to 15 degrees does what?",
         {"A": "Leaves it unchanged, since the number of bonds has not changed", "B": "Doubles it, since each bond now has twice as many settings to try", "C": "Multiplies it by two to the power of twelve, about four thousandfold", "D": "Halves it, since finer steps make the search converge much sooner"},
         "C", {"B": "Doubling applies to one bond; with twelve bonds the factor is 2 to the twelfth, about 4,000-fold.", "D": "Finer sampling always increases the number of points to evaluate.", "A": "The count is settings raised to the power of bonds, so settings matter as much as bonds.", "_correct": "The count is (360/step) raised to the number of bonds. Halving the step multiplies the total by 2 to the power of the bond count, which for twelve bonds is about a 4,000-fold increase. This is why systematic search is abandoned in favour of stochastic methods."},
         bloom="Apply", silo="6", section="concept"),
    Item("Q1", "A conformational search returns a structure the program labels a minimum. What has been established about it?",
         {"A": "It is the most stable conformation the molecule can adopt", "B": "It is the conformation observed in the crystal structure", "C": "It is the conformation the molecule adopts when it binds", "D": "It is a stable form, not necessarily the lowest in energy"},
         "D", {"A": "is the local/global confusion", "C": "assumes most stable equals active", "B": "conflates calculation with experiment", "_correct": "A minimum only means every small change raises the energy; establishing that it is the lowest requires searching the whole surface."},
         bloom="Understand", silo="6", section="post"),
    Item("Q2", "Which feature distinguishes a Monte Carlo search from molecular dynamics?",
         {"A": "It follows Newton's equations to produce a time trajectory", "B": "It generates structures by random rotation about single bonds", "C": "It encodes each torsion as a gene and recombines the strings", "D": "It rotates each rotatable bond through fixed equal increments"},
         "B", {"C": "describes a genetic algorithm", "D": "describes stepwise rotation", "A": "describes MD", "_correct": "Monte Carlo makes random changes to rotatable bonds and tests each one, rather than integrating motion over time as dynamics does."},
         bloom="Understand", silo="6", section="post"),
    Item("Q3", "In simulated annealing, why is the Monte Carlo run repeated at progressively lower temperatures?",
         {"A": "Acceptance becomes stricter, settling the search in deep minima", "B": "Cooling physically strengthens the bonds in the stable conformer", "C": "Low temperatures let the program sample more diverse structures", "D": "Each cycle removes one rotatable bond from the search problem"},
         "A", {"B": "treats a numerical schedule as physical", "D": "invents a mechanism", "C": "inverts the role of temperature", "_correct": "Lowering the temperature reduces the chance of accepting an uphill move, so the search is progressively confined to the deepest well it has found."},
         bloom="Understand", silo="6", section="post"),
    Item("Q4", "Two analogues are overlaid and the program reports RMSD = 0.35 A. What does that number describe?",
         {"A": "The difference in steric energy between the two conformations", "B": "The predicted difference in binding affinity for the two ligands", "C": "The average separation of the matched atom pairs after fitting", "D": "The proportion of molecular volume shared by the two structures"},
         "C", {"A": "RMSD is a geometric distance measure and carries no steric energy", "B": "RMSD reports geometry only, so it contains no affinity information", "D": "Shared volume is a volume-overlap measure, not a distance between matched atoms", "_correct": "RMSD is the root mean square of the distances between matched atom pairs after the best fit, so it is a purely geometric quantity."},
         bloom="Understand", silo="6", section="post"),
    Item("Q5", "Why is a rigid active analogue the preferred starting point for defining a 3D pharmacophore?",
         {"A": "Rigid molecules are usually more potent than flexible ones", "B": "Its crystal structure can be solved without a protein present", "C": "Rigid structures are easier to synthesize in useful quantity", "D": "Its few conformations fix the geometry of the binding groups"},
         "D", {"A": "is an unsupported potency claim", "C": "is an irrelevant practical appeal", "B": "is true of many compounds but not the reason", "_correct": "A rigid analogue has few accessible conformations, so the geometry of its binding groups can be taken as the pharmacophore geometry."},
         bloom="Understand", silo="6", section="post"),
]

gb = Gradebook("L23", "Conformational Analysis I")
gb.add_many(ITEMS)
print("Ready - 15 questions loaded. Scroll down and begin.")

# ===== BEGIN structure-drawing and calculator kit (lecture notes) =====
# ---------------------------------------------------------------------------
# PHS 511 notebook helper library (pure Python + matplotlib; no RDKit, no network)
# Draws textbook-oriented 2D skeletal structures from explicit coordinates so
# every figure has a fixed, reviewed orientation:
#   * acyl chains: C1 (carbonyl) on the LEFT, omega (methyl) end on the RIGHT
#   * glycerol / sphingosine backbones vertical (sn-1 top, sn-3 bottom)
#   * steroids: rings A-B-C-D left to right, C3-OH lower left, C18/C19 up
# ---------------------------------------------------------------------------
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, FancyBboxPatch, Circle

COL = {
    "ink": "#1f2430", "muted": "#6b7280",
    "ester": "#1d6fb8", "amide": "#c2410c", "ether": "#7c3aed",
    "phos": "#15803d", "oh": "#b91c1c", "dbl": "#d97706",
    "delta": "#1d6fb8", "omega": "#b91c1c", "drug": "#be123c",
    "polar": "#0e7490", "nonpolar": "#a16207", "hl": "#fde68a",
}
S30, C30 = 0.5, math.cos(math.radians(30))
UNIT_IN = 0.36   # inches per bond length -> identical scale in every figure


class Mol:
    """Atoms with explicit 2D coordinates; bonds with order/style/highlight."""

    def __init__(self, name=""):
        self.name, self.atoms, self.bonds, self.notes, self.boxes = name, [], [], [], []

    def add(self, x, y, label="", color=None, ha="center", size=None, anchor=None):
        """anchor: index of the label character that sits on the atom (default: first C/N/O/P/S)."""
        self.atoms.append(dict(x=x, y=y, label=label, color=color, ha=ha, size=size, anchor=anchor))
        return len(self.atoms) - 1

    def bond(self, i, j, order=1, style="plain", hl=None, color=None, side=None):
        self.bonds.append(dict(i=i, j=j, order=order, style=style, hl=hl, color=color, side=side))
        return len(self.bonds) - 1

    def note(self, x, y, text, color=None, size=8.5, ha="center", va="center", weight="normal"):
        self.notes.append(dict(x=x, y=y, text=text, color=color, size=size, ha=ha, va=va, weight=weight))

    def pos(self, i):
        a = self.atoms[i]
        return a["x"], a["y"]



def _turtle(m, start, n, d0, turns, orders, hl=None):
    idx = [start]
    x, y = m.pos(start)
    d = d0
    for k in range(1, n + 1):
        x, y = x + math.cos(math.radians(d)), y + math.sin(math.radians(d))
        idx.append(m.add(x, y))
        m.bond(idx[-2], idx[-1], order=orders.get(k, 1), hl=hl)
        if k < n:
            d += turns[k]
    return idx


def chain(m, start, n, heading=0.0, first=+1, cis=(), trans=(), hl=None, flip=()):
    """Zigzag chain of n carbons grown from atom `start` (start = carbon 0).
    Bond k joins chain carbons k-1 and k. The turn at carbon k normally
    alternates sign (zigzag). If bond k is a cis C=C, the turn at carbon k
    repeats the previous sign so both substituents sit on the same side.
    `flip` lists carbons where a single bond is drawn rotated (same-sign turn)
    to steer a long chain back into its row; it does not change the molecule."""
    turns, orders = {}, {}
    sign = -first
    for k in range(1, n):
        if k >= 2:
            sign = sign if (k in cis or k in flip) else -sign
        turns[k] = 60 * sign
    for k in list(cis) + list(trans):
        orders[k] = 2
    return _turtle(m, start, n, heading + 30 * first, turns, orders, hl)


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
SUBS = set("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u207a\u207b\u00b9\u00b2\u00b3\u2074")
EM = 1.0 / (72 * UNIT_IN)   # data units per point at the fixed drawing scale


def _cw(ch):
    """Approximate glyph advance (em) for DejaVu Sans."""
    if ch in SUBS:
        return 0.40
    if ch in "()":
        return 0.39
    if ch.isupper():
        return 0.72
    if ch.isdigit():
        return 0.64
    return 0.56


def _label_box(lab, fs, anchor=None):
    """(left, right, half-height) of a label in data units, measured from the
    anchor glyph centre (first C/N/O/P/S), which sits exactly on the atom."""
    em = fs * EM
    ws = [_cw(c) * em for c in lab]
    k = anchor if anchor is not None else next((i for i, c in enumerate(lab) if c in "CNOPS"), 0)
    left = sum(ws[:k]) + ws[k] / 2
    right = sum(ws[k + 1:]) + ws[k] / 2
    return left, right, 0.36 * fs * EM * 1.0


def _label_radius(lab, ux, uy, fs=10.5, anchor=None):
    """Distance from atom centre to the label box edge along (ux, uy), plus a gap."""
    if not lab:
        return 0.0
    left, right, hh = _label_box(lab, fs, anchor)
    tx = (right if ux > 0 else left) / abs(ux) if abs(ux) > 1e-9 else 1e9
    ty = hh / abs(uy) if abs(uy) > 1e-9 else 1e9
    return min(tx, ty) + 0.08


def draw(ax, m, fs=10.5):
    A = m.atoms
    nb = {i: [] for i in range(len(A))}
    for b in m.bonds:
        nb[b["i"]].append(b["j"]); nb[b["j"]].append(b["i"])
    # highlights first (under everything)
    for b in m.bonds:
        if b["hl"]:
            (x1, y1), (x2, y2) = m.pos(b["i"]), m.pos(b["j"])
            ax.plot([x1, x2], [y1, y2], color=b["hl"], lw=9, alpha=0.30,
                    solid_capstyle="round", zorder=1)
    for b in m.bonds:
        i, j = b["i"], b["j"]
        (x1, y1), (x2, y2) = m.pos(i), m.pos(j)
        L = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / L, (y2 - y1) / L
        r1 = _label_radius(A[i]["label"], ux, uy, A[i]["size"] or fs, A[i]["anchor"])
        r2 = _label_radius(A[j]["label"], ux, uy, A[j]["size"] or fs, A[j]["anchor"])
        ax1, ay1 = x1 + ux * r1, y1 + uy * r1
        ax2, ay2 = x2 - ux * r2, y2 - uy * r2
        c = b["color"] or COL["ink"]
        st = b["style"]
        if st == "wedge":
            px, py = -uy * 0.11, ux * 0.11
            ax.add_patch(Polygon([(ax1, ay1), (ax2 + px, ay2 + py), (ax2 - px, ay2 - py)],
                                 closed=True, fc=c, ec=c, lw=0.6, zorder=3))
            continue
        if st == "hash":
            nseg = 7
            for s in range(nseg + 1):
                f = s / nseg
                cx, cy = ax1 + (ax2 - ax1) * f, ay1 + (ay2 - ay1) * f
                w = 0.02 + 0.10 * f
                ax.plot([cx - uy * w, cx + uy * w], [cy + ux * w, cy - ux * w],
                        color=c, lw=1.1, zorder=3, solid_capstyle="butt")
            continue
        if st == "wavy":
            n = 40
            xs = [ax1 + (ax2 - ax1) * k / n - uy * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ys = [ay1 + (ay2 - ay1) * k / n + ux * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ax.plot(xs, ys, color=c, lw=1.4, zorder=3)
            continue
        px, py = -uy, ux
        if b["order"] == 1:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round",
                    ls="--" if st == "dashed" else "-")
            continue
        side = b["side"]
        if side is None:
            if A[i]["label"] or A[j]["label"]:
                side = 0                      # C=O, C=N+: centred pair
            else:
                sx = sy = 0.0
                for k in nb[i]:
                    if k != j:
                        sx += A[k]["x"] - x1; sy += A[k]["y"] - y1
                for k in nb[j]:
                    if k != i:
                        sx += A[k]["x"] - x2; sy += A[k]["y"] - y2
                cross = ux * sy - uy * sx
                side = 1 if cross >= -1e-6 else -1
        if side == 0:
            for sgn in (+1, -1):
                o = 0.075 * sgn
                ax.plot([ax1 + px * o, ax2 + px * o], [ay1 + py * o, ay2 + py * o],
                        color=c, lw=1.5, zorder=3, solid_capstyle="round")
        else:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round")
            o, sh = 0.15 * side, 0.16
            ax.plot([ax1 + px * o + ux * sh, ax2 + px * o - ux * sh],
                    [ay1 + py * o + uy * sh, ay2 + py * o - uy * sh],
                    color=c, lw=1.5, zorder=3, solid_capstyle="round")
    for a in A:
        if a["label"]:
            l_, r_, _ = _label_box(a["label"], a["size"] or fs, a["anchor"])
            ax.text(a["x"] + (r_ - l_) / 2, a["y"], a["label"], ha="center", va="center",
                    fontsize=a["size"] or fs, color=a["color"] or COL["ink"],
                    zorder=5, fontweight="normal")
    for n_ in m.notes:
        ax.text(n_["x"], n_["y"], n_["text"], ha=n_["ha"], va=n_["va"], fontsize=n_["size"],
                color=n_["color"] or COL["muted"], zorder=6, fontweight=n_["weight"])


def extent(m, pad=0.7):
    xs = [a["x"] for a in m.atoms] + [n["x"] for n in m.notes]
    ys = [a["y"] for a in m.atoms] + [n["y"] for n in m.notes]
    return min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad




def show(m, title=None, unit=UNIT_IN, extra=None, caption=None, show_now=True):
    """One structure at the fixed scale (UNIT_IN inches per bond)."""
    x0, x1, y0, y1 = extent(m)
    if extra and hasattr(extra, "bounds"):
        bx0, bx1, by0, by1 = extra.bounds
        x0, x1, y0, y1 = min(x0, bx0), max(x1, bx1), min(y0, by0), max(y1, by1)
    th = 0.42 if title else 0.0
    ch = 0.40 if caption else 0.0
    W, H = (x1 - x0) * unit, (y1 - y0) * unit
    fig = plt.figure(figsize=(W, H + th + ch))
    ax = fig.add_axes([0, ch / (H + th + ch), 1, H / (H + th + ch)])
    draw(ax, m)
    if extra:
        extra(ax)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        fig.text(0.0, 1 - (th / 2) / (H + th + ch), title, fontsize=11.5, fontweight="bold",
                 color=COL["ink"], va="center", ha="left")
    if caption:
        fig.text(0.0, (ch / 2) / (H + th + ch), caption, fontsize=9, color=COL["muted"],
                 va="center", ha="left")
    if show_now:
        plt.show()
    return fig


def grid(mols, titles, ncols=2, unit=UNIT_IN, align="left", suptitle=None, captions=None, show_now=True, bottom=0.0):
    """Panels drawn at one common scale; each panel box has the same width and
    the row height of its tallest member, so structures line up cleanly."""
    exts = [extent(m) for m in mols]
    W = max(max(e[1] - e[0] for e in exts), 3.6 / unit)   # panels at least 3.6 in wide
    nrows = math.ceil(len(mols) / ncols)
    import textwrap as _tw
    maxch = max(12, int(W * unit * 72 / (10.5 * 0.60)) - 1)
    wrapped = [_tw.fill(t, maxch) for t in titles]
    rowH = []
    for r in range(nrows):
        rowH.append(max(exts[k][3] - exts[k][2] for k in range(r * ncols, min(len(mols), (r + 1) * ncols))))
    row_th = []
    for r in range(nrows):
        nl = max(wrapped[k].count("\n") + 1 for k in range(r * ncols, min(len(mols), (r + 1) * ncols)))
        row_th.append(0.27 + 0.19 * nl)
    fig_w = W * unit * ncols
    fig_h = sum(h * unit + th for h, th in zip(rowH, row_th)) + (0.5 if suptitle else 0) + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    top = 1.0 - (0.5 / fig_h if suptitle else 0)
    y_cursor = top
    for r in range(nrows):
        h_frac = (rowH[r] * unit) / fig_h
        t_frac = row_th[r] / fig_h
        y_cursor -= (h_frac + t_frac)
        for c in range(ncols):
            k = r * ncols + c
            if k >= len(mols):
                break
            ax = fig.add_axes([c / ncols, y_cursor, 1 / ncols, h_frac])
            m = mols[k]
            x0, x1, y0, y1 = exts[k]
            draw(ax, m)
            if align == "left":
                ax.set_xlim(x0, x0 + W)
            else:
                cx = (x0 + x1) / 2
                ax.set_xlim(cx - W / 2, cx + W / 2)
            cy = (y0 + y1) / 2
            ax.set_ylim(cy - rowH[r] / 2, cy + rowH[r] / 2)
            ax.set_aspect("equal"); ax.axis("off")
            ax.text(0.01, 1.0 + 0.02, wrapped[k], transform=ax.transAxes, fontsize=10.5, linespacing=1.15,
                    fontweight="bold", color=COL["ink"], va="bottom", ha="left")
    if suptitle:
        fig.text(0.005, 1 - 0.25 / fig_h, suptitle, fontsize=12.5, fontweight="bold", va="center")
    if show_now:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Structure builders (coordinates chosen to match textbook orientation)
# ---------------------------------------------------------------------------
def _outer(m, i, nbrs, dist=0.55):
    """Point on the open (outer) side of atom i, away from its neighbours."""
    x, y = m.pos(i)
    sx = sum(m.atoms[k]["x"] - x for k in nbrs); sy = sum(m.atoms[k]["y"] - y for k in nbrs)
    L = math.hypot(sx, sy) or 1.0
    return x - sx / L * dist, y - sy / L * dist


def fatty_acid(n, cis=(), trans=(), name="", numbers=None, ionized=False, origin=(0.0, 0.0),
               hl_double=True, flip=()):
    """Free fatty acid, C1 (COOH) on the LEFT, methyl (omega) end on the RIGHT.
    cis/trans: Delta positions (bond between C_p and C_p+1).
    numbers: None | 'delta' | 'omega' | 'both'"""
    m = Mol(name)
    ox, oy = origin
    c1 = m.add(ox, oy)
    o_dbl = m.add(ox, oy + 1.0, "O")
    o_h = m.add(ox - C30, oy - 0.5, "⁻O" if ionized else "HO")
    m.bond(c1, o_dbl, 2); m.bond(c1, o_h, 1)
    hl = COL["hl"] if hl_double else None
    ids = chain(m, c1, n - 1, heading=0, first=-1, cis=cis, trans=trans, flip=flip)
    if hl_double:
        for b in m.bonds:
            if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
                b["hl"] = COL["dbl"]
    m.carbons = ids  # carbons[0] = C1
    if numbers:
        dbs = sorted(list(cis) + list(trans))
        last = dbs[-1] if dbs else n - 2
        for p, a in enumerate(ids, start=1):
            if p == 1:
                nb = [o_dbl, o_h, ids[1]]
            elif p == n:
                nb = [ids[p - 2]]
            else:
                nb = [ids[p - 2], ids[p]]
            if numbers in ("delta", "both"):
                x, y = _outer(m, a, nb, 0.50)
                m.note(x, y, str(p), color=COL["delta"], size=7.5)
            w = n - p + 1
            if numbers == "omega" or (numbers == "both" and p >= last):
                if p == n:
                    x, y = _outer(m, a, nb, 0.50)
                    y -= 0.42 if numbers == "both" else 0.0
                else:
                    x, y = _outer(m, a, nb, 0.50 if numbers == "omega" else -0.62)
                m.note(x, y, ("\u03c9" + str(w)) if w == 1 else str(w), color=COL["omega"], size=7.5,
                       weight="bold")
    return m


def acyl_row(m, o_atom, n, cis=(), trans=(), carbonyl_up=True, flip=None):
    """Acyl group C(=O)-chain attached to ester/amide atom o_atom, chain to the right."""
    x, y = m.pos(o_atom)
    c1 = m.add(x + C30, y + (0.5 if carbonyl_up else -0.5))
    m.bond(o_atom, c1, 1)
    od = m.add(x + C30, y + (1.5 if carbonyl_up else -1.5), "O")
    m.bond(c1, od, 2)
    if flip is None:   # keep long chains in their row: rotate the bond 3 carbons past the last cis C=C
        flip = tuple(p + 2 for p in cis)
    ids = chain(m, c1, n - 1, heading=0, first=-1 if carbonyl_up else +1, cis=cis, trans=trans, flip=flip)
    for b in m.bonds:
        if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
            b["hl"] = COL["dbl"]
    return c1, od, ids


ROW = 2.6  # vertical spacing of backbone carbons in E-shaped lipids


def backbone(m, labels, x=0.0, y0=0.0):
    """Vertical 3-carbon backbone (glycerol or sphingosine C1-C3), top->bottom."""
    ids = []
    for k, lab in enumerate(labels):
        ids.append(m.add(x, y0 - k * ROW, lab))
        if k:
            m.bond(ids[k - 1], ids[k], 1)
    return ids


def ester(m, c_backbone, n, cis=(), trans=(), hl=True, cut=None):
    x, y = m.pos(c_backbone)
    o = m.add(x + 1.15, y, "O")
    b0 = m.bond(c_backbone, o, 1, hl=COL["ester"] if hl else None)
    c1, od, ids = acyl_row(m, o, n, cis, trans)
    m.bonds[-(len(ids)) - 1]["hl"] = None
    # highlight O-C(=O) (the ester bond lipases cut)
    for b in m.bonds:
        if b["i"] == o and b["j"] == c1:
            b["hl"] = COL["ester"] if hl else None
            if cut:
                b["color"] = COL["drug"]
    return o, c1, ids


def triacylglycerol(acyls=((16, ()), (18, (9,)), (18, (9, 12))), name="Triacylglycerol",
                    sn=True, cuts=None):
    """E-shaped TAG; glycerol vertical, sn-1 top. acyls: ((n, cis-positions), ...)."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    rows = []
    for k, (n, cis) in enumerate(acyls):
        rows.append(ester(m, g[k], n, cis=cis))
    if sn:
        for k in range(3):
            x, y = m.pos(g[k])
            m.note(x - 0.95, y, "sn-%d" % (k + 1), color=COL["muted"], size=8.5, ha="right")
    m.g, m.rows = g, rows
    return m


def mark_cut(m, o_idx, c_idx, text, color=None, dy=-0.62, dx=0.0):
    x1, y1 = m.pos(o_idx); x2, y2 = m.pos(c_idx)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my - 0.05, "✂", color=color or COL["drug"], size=13)
    m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5, ha="left" if dx else "center")


def phosphocholine(m, c_attach, head="choline"):
    """-O-P(=O)(O-)-O-CH2-CH2-N+(CH3)3 to the right of c_attach (horizontal)."""
    x, y = m.pos(c_attach)
    o1 = m.add(x + 1.15, y, "O"); m.bond(c_attach, o1, 1)
    p = m.add(x + 2.3, y, "P", color=COL["phos"]); m.bond(o1, p, 1, hl=COL["phos"])
    po = m.add(x + 2.3, y + 1.05, "O"); m.bond(p, po, 2)
    pm = m.add(x + 2.3, y - 1.05, "O⁻"); m.bond(p, pm, 1)
    o2 = m.add(x + 3.45, y, "O"); m.bond(p, o2, 1, hl=COL["phos"])
    if head == "choline":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "N⁺(CH₃)₃"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.6, y - 1.0, "choline", color=COL["muted"], size=8)
    elif head == "ethanolamine":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "NH₃⁺"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.4, y - 1.0, "ethanolamine", color=COL["muted"], size=8)
    return p


def phosphatidylcholine(name="Phosphatidylcholine (lecithin)", a1=(16, ()), a2=(18, (9,))):
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    ester(m, g[0], *a1); ester(m, g[1], *a2)
    phosphocholine(m, g[2], "choline")
    for k in range(3):
        x, y = m.pos(g[k]); m.note(x - 0.95, y, "sn-%d" % (k + 1), size=8.5, ha="right")
    m.g = g
    return m


def plasmalogen(name="Plasmalogen (ethanolamine)", n1=16, a2=(18, (9,))):
    """sn-1 vinyl ether: CH2-O-CH=CH-R (cis), no carbonyl."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    x, y = m.pos(g[0])
    o = m.add(x + 1.15, y, "O"); m.bond(g[0], o, 1, hl=COL["ether"])
    ids = chain(m, o, n1, heading=0, first=+1, cis=(2,), flip=(4,))
    # bond O-C1' is ether; C1'=C2' vinyl (cis, as in natural plasmalogens)
    for b in m.bonds:
        if b["i"] == o and b["j"] == ids[1]:
            b["hl"] = COL["ether"]
        if b["order"] == 2:
            b["hl"] = COL["ether"]
    ester(m, g[1], *a2)
    phosphocholine(m, g[2], "ethanolamine")
    for k in range(3):
        xx, yy = m.pos(g[k]); m.note(xx - 0.95, yy, "sn-%d" % (k + 1), size=8.5, ha="right")
    return m


def sphingo_core(m, fa_n=18, fa_cis=()):
    """Sphingosine drawn as the same vertical 3-carbon backbone:
    C3 top (CH-OH + trans CH=CH tail), C2 middle (CH-NH-acyl, amide), C1 bottom (CH2-O-head)."""
    s = backbone(m, ["CH", "CH", "CH₂"])
    x, y = m.pos(s[0])
    oh = m.add(x - 1.15, y, "HO"); m.bond(s[0], oh, 1, hl=COL["oh"])
    ids = chain(m, s[0], 15, heading=0, first=+1, trans=(1,))
    # C4=C5 trans: chain bond 1 is C3-C4? -> rebuild explicitly
    m.bonds[[k for k, b in enumerate(m.bonds) if b["i"] == s[0] and b["j"] == ids[1]][0]]["order"] = 1
    for b in m.bonds:
        if b["i"] == ids[1] and b["j"] == ids[2]:
            b["order"] = 2; b["hl"] = COL["dbl"]
    # amide at C2
    x2, y2 = m.pos(s[1])
    nh = m.add(x2 + 1.4, y2, "NH"); m.bond(s[1], nh, 1)
    c1, od, acyl = acyl_row(m, nh, fa_n, cis=fa_cis)
    for b in m.bonds:
        if b["i"] == nh and b["j"] == c1:
            b["hl"] = COL["amide"]
    for k, lab in enumerate(["C3", "C2", "C1"]):
        xx, yy = m.pos(s[k]); m.note(xx - 0.25, yy - 0.55, lab, size=7.5)
    return s


def sphingomyelin(name="Sphingomyelin"):
    m = Mol(name)
    s = sphingo_core(m, 18)
    phosphocholine(m, s[2], "choline")
    return m


def galactosylceramide(name="Galactosylceramide (a cerebroside)"):
    m = Mol(name)
    s = sphingo_core(m, 24, fa_cis=(15,))
    x, y = m.pos(s[2])
    o = m.add(x + 1.15, y, "O"); m.bond(s[2], o, 1, hl=COL["polar"])
    # beta-D-galactopyranose, Haworth-free chair-less hexagon (ring O upper right)
    cx, cy, r = x + 3.0, y - 0.1, 1.0
    ang = [180, 120, 60, 0, -60, -120]
    pts = [m.add(cx + r * math.cos(math.radians(a)), cy + r * math.sin(math.radians(a)),
                 "O" if k == 1 else "") for k, a in enumerate(ang)]
    for k in range(6):
        m.bond(pts[k], pts[(k + 1) % 6])
    m.bond(o, pts[0], 1, hl=COL["polar"])
    m.note(cx, cy - 0.05, "Gal", color=COL["polar"], size=9, weight="bold")
    return m


# --- steroids --------------------------------------------------------------
def _hex(cx, cy):
    return [(cx + math.cos(math.radians(a)), cy + math.sin(math.radians(a))) for a in (90, 30, -30, -90, -150, 150)]


def steroid_core(m, ox=0.0, oy=0.0, dbl=()):
    """Gonane nucleus in standard orientation. Returns dict C1..C17 -> atom index.
    Unit hexagon geometry with vertical shared A/B edge (C5-C10)."""
    r3 = math.sqrt(3)
    P = {}
    A = _hex(ox, oy)                      # top, ur, lr, bottom, ll, ul
    P.update({1: A[0], 10: A[1], 5: A[2], 4: A[3], 3: A[4], 2: A[5]})
    B = _hex(ox + r3, oy)
    P.update({9: B[0], 8: B[1], 6: B[3], 7: B[2]})
    C = _hex(ox + 1.5 * r3, oy + 1.5)
    P.update({12: C[0], 13: C[1], 14: C[2], 11: C[5]})
    # ring D: regular pentagon on edge C13-C14, outward (to the right)
    (x13, y13), (x14, y14) = P[13], P[14]
    mx, my = (x13 + x14) / 2, (y13 + y14) / 2
    s = 1.0
    apo = s / (2 * math.tan(math.radians(36)))
    cx, cy = mx + apo, my
    R = s / (2 * math.sin(math.radians(36)))
    base = math.degrees(math.atan2(y13 - cy, x13 - cx))
    pent = [(cx + R * math.cos(math.radians(base - 72 * k)), cy + R * math.sin(math.radians(base - 72 * k)))
            for k in range(5)]
    # pent[0]=C13, going clockwise: C17, C16, C15, C14
    P.update({17: pent[1], 16: pent[2], 15: pent[3]})
    idx = {k: m.add(v[0], v[1]) for k, v in P.items()}
    ring_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 10), (10, 1), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
                  (9, 11), (11, 12), (12, 13), (13, 14), (14, 8), (13, 17), (17, 16), (16, 15), (15, 14)]
    for a, b in ring_bonds:
        m.bond(idx[a], idx[b], 2 if (a, b) in dbl or (b, a) in dbl else 1)
    m.st = idx
    return idx


def subst(m, at, dx, dy, label="", style="plain", order=1, hl=None, color=None):
    x, y = m.pos(at)
    k = m.add(x + dx, y + dy, label, color=color)
    m.bond(at, k, order, style=style, hl=hl)
    return k


def side_chain_cholestane(m, st, extra_ethyl=False):
    """C20-C27 side chain on C17 (drawn up-right, zigzag)."""
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 4, heading=0, first=-1)
    c25 = ids[-1]
    x25, y25 = m.pos(c25)
    c26 = m.add(x25 + C30, y25 + 0.5); m.bond(c25, c26)
    c27 = m.add(x25, y25 - 1.0); m.bond(c25, c27)
    if extra_ethyl:
        x24, y24 = m.pos(ids[3])
        ex, ey = _outer(m, ids[3], [ids[2], ids[4]], 1.0)
        e1 = m.add(ex, ey); m.bond(ids[3], e1, hl=COL["nonpolar"])
        e2 = m.add(ex + C30, ey + (0.5 if ey > y24 else -0.5)); m.bond(e1, e2, hl=COL["nonpolar"])
    return ids


def cholesterol(name="Cholesterol", sitosterol=False):
    m = Mol(name)
    st = steroid_core(m, dbl=((5, 6),))
    ho = subst(m, st[3], -C30, -0.5, "HO", style="wedge", hl=COL["oh"])
    subst(m, st[10], 0.0, 0.95, "", style="wedge")     # C19 methyl
    subst(m, st[13], 0.0, 0.95, "", style="wedge")     # C18 methyl
    side_chain_cholestane(m, st, extra_ethyl=sitosterol)
    m.note(m.pos(st[3])[0] - 0.2, m.pos(st[3])[1] + 0.45, "3β", size=7.5)
    for k, lab in ((1, "A"), (2, "B"), (3, "C")):
        pass
    r3 = math.sqrt(3)
    for (cx, cy, lab) in ((0, 0, "A"), (r3, 0, "B"), (1.5 * r3, 1.5, "C")):
        m.note(cx, cy, lab, color="#9aa3b2", size=9, weight="bold")
    x13, y13 = m.pos(st[13]); x14, y14 = m.pos(st[14])
    m.note((x13 + x14) / 2 + 0.68, (y13 + y14) / 2, "D", color="#9aa3b2", size=9, weight="bold")
    m.st = st
    return m


def bile_acid(name="Glycocholate", oh=(("3", "a"), ("7", "a"), ("12", "a")), conj="glycine", ionized=True):
    """5-beta bile acid; oh entries (position, 'a' alpha=hash | 'b' beta=wedge)."""
    m = Mol(name)
    st = steroid_core(m)
    subst(m, st[10], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[13], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[5], 0.0, -0.85, "H", style="wedge")      # 5-beta H (A/B cis)
    vec = {"3": (-C30, -0.5), "7": (C30 * 0.0, -1.0), "12": (0.0, 1.0)}
    for pos, face in oh:
        dx, dy = vec[pos]
        if pos == "12":
            dx, dy = (-0.35, 0.9)
        if pos == "7":
            dx, dy = (0.35, -0.95)
        subst(m, st[int(pos)], dx, dy, "HO" if pos == "3" else "OH",
              style="hash" if face == "a" else "wedge", hl=COL["polar"])
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 3, heading=0, first=-1)
    c24 = ids[-1]
    x24, y24 = m.pos(c24)
    o24 = m.add(x24, y24 + 1.0, "O"); m.bond(c24, o24, 2)
    if conj == "glycine":
        nh = m.add(x24 + C30, y24 - 0.5, "NH"); m.bond(c24, nh, 1, hl=COL["amide"])
        ca = m.add(x24 + 2 * C30, y24); m.bond(nh, ca)
        cc = m.add(x24 + 3 * C30, y24 - 0.5); m.bond(ca, cc)
        m.bond(cc, m.add(x24 + 3 * C30, y24 - 1.5, "O"), 2)
        m.bond(cc, m.add(x24 + 4 * C30, y24, "O⁻" if ionized else "OH"), 1, hl=COL["polar"])
    else:
        m.bond(c24, m.add(x24 + C30, y24 - 0.5, "OH"), 1, hl=COL["polar"])
    m.st = st
    return m


# ---------------------------------------------------------------------------
# General 2D sketch helpers: rings, fused rings, substituents, reaction schemes
# (built on draw.py; every coordinate is explicit so orientation is fixed)
# ---------------------------------------------------------------------------
def _nbrs(m, i):
    out = []
    for b in m.bonds:
        if b["i"] == i:
            out.append(b["j"])
        elif b["j"] == i:
            out.append(b["i"])
    return out


def bond_between(m, i, j):
    for b in m.bonds:
        if (b["i"], b["j"]) in ((i, j), (j, i)):
            return b
    return None


def ring(m, cx, cy, n=6, rot=90.0, labels=None, side=1.0):
    """Regular n-ring with bond length `side`, first vertex at angle `rot` (deg),
    vertices counter-clockwise. labels: list of n labels ('' = carbon).
    Returns atom indices in ring order."""
    R = side / (2 * math.sin(math.pi / n))
    ids = []
    for k in range(n):
        a = math.radians(rot + 360.0 * k / n)
        ids.append(m.add(cx + R * math.cos(a), cy + R * math.sin(a), (labels[k] if labels else "")))
    for k in range(n):
        m.bond(ids[k], ids[(k + 1) % n], 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def fuse(m, a, b, n=6, labels=None, side=1.0):
    """New regular n-ring sharing edge a-b, built on the side away from the
    existing neighbours. labels: for the n-2 NEW atoms, in order from b onward.
    Returns ring atoms [a, b, new...]."""
    (xa, ya), (xb, yb) = m.pos(a), m.pos(b)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ex, ey = xb - xa, yb - ya
    L = math.hypot(ex, ey)
    px, py = -ey / L, ex / L
    # decide which side is free: away from centroid of current neighbours
    nb = [k for k in _nbrs(m, a) + _nbrs(m, b) if k not in (a, b)]
    if nb:
        cx_ = sum(m.atoms[k]["x"] for k in nb) / len(nb); cy_ = sum(m.atoms[k]["y"] for k in nb) / len(nb)
        if (cx_ - mx) * px + (cy_ - my) * py > 0:
            px, py = -px, -py
    apo = side / (2 * math.tan(math.pi / n))
    cx, cy = mx + px * apo, my + py * apo
    R = side / (2 * math.sin(math.pi / n))
    # walk from b around the new ring (away from a)
    ang_b = math.atan2(yb - cy, xb - cx)
    ang_a = math.atan2(ya - cy, xa - cx)
    step = 2 * math.pi / n
    # direction: going from b, the next vertex must not be a
    d = (ang_b - ang_a + math.pi) % (2 * math.pi) - math.pi
    sgn = 1 if d > 0 else -1
    ids = [a, b]
    prev = b
    for k in range(1, n - 1):
        ang = ang_b + sgn * step * k
        lab = labels[k - 1] if labels else ""
        i = m.add(cx + R * math.cos(ang), cy + R * math.sin(ang), lab)
        m.bond(prev, i, 1)
        ids.append(i); prev = i
    m.bond(prev, a, 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def set_double(m, pairs):
    """pairs: iterable of (i, j) atom-index pairs to make double bonds."""
    for i, j in pairs:
        b = bond_between(m, i, j)
        if b is None:
            raise ValueError("no bond %d-%d" % (i, j))
        b["order"] = 2


def kekule(m, ids, start=0):
    """Alternate double bonds around ring `ids` beginning with bond start->start+1."""
    n = len(ids)
    for k in range(start, start + n, 2):
        bond_between(m, ids[k % n], ids[(k + 1) % n])["order"] = 2


def outward(m, i, dist=1.0):
    """Unit outward direction from atom i (away from its neighbours)."""
    x, y = m.pos(i)
    nb = _nbrs(m, i)
    sx = sum(m.atoms[k]["x"] - x for k in nb); sy = sum(m.atoms[k]["y"] - y for k in nb)
    L = math.hypot(sx, sy)
    if L < 1e-9:
        return (dist, 0.0)
    return (-sx / L * dist, -sy / L * dist)


def sub(m, i, label="", angle=None, dist=1.0, order=1, style="plain", hl=None, color=None, anchor=None):
    """Attach a substituent to atom i: outward by default, or at an absolute angle (deg)."""
    x, y = m.pos(i)
    if angle is None:
        dx, dy = outward(m, i, dist)
    else:
        dx, dy = dist * math.cos(math.radians(angle)), dist * math.sin(math.radians(angle))
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, style=style, hl=hl)
    return k


def grow(m, i, n, angle=None, first=+1, cis=(), trans=(), flip=()):
    """Zigzag chain of n atoms from atom i; heading = outward unless angle given."""
    if angle is None:
        dx, dy = outward(m, i)
        angle = math.degrees(math.atan2(dy, dx))
    return chain(m, i, n, heading=angle, first=first, cis=cis, trans=trans, flip=flip)


def label(m, i, text, anchor=None):
    m.atoms[i]["label"] = text
    m.atoms[i]["anchor"] = anchor


def hl_bonds(m, pairs, color):
    for i, j in pairs:
        bond_between(m, i, j)["hl"] = color


def hl_atoms_note(m, i, text, color=None, dist=0.75, size=8):
    x, y = m.pos(i)
    dx, dy = outward(m, i, dist)
    m.note(x + dx, y + dy, text, color=color or COL["muted"], size=size)


def shift(m, dx, dy):
    for a in m.atoms:
        a["x"] += dx; a["y"] += dy
    for n in m.notes:
        n["x"] += dx; n["y"] += dy
    return m


def rotate(m, deg, about=None):
    """Rotate a whole drawing (use to fix an orientation, e.g. to put a ring flat)."""
    if about is None:
        xs = [a["x"] for a in m.atoms]; ys = [a["y"] for a in m.atoms]
        about = ((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2)
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    for obj in m.atoms + m.notes:
        x, y = obj["x"] - about[0], obj["y"] - about[1]
        obj["x"], obj["y"] = about[0] + c * x - s * y, about[1] + s * x + c * y
    return m


def mirror(m, axis="x"):
    """Mirror left-right (axis='x') or top-bottom ('y'); wedges keep their sense."""
    for obj in m.atoms + m.notes:
        if axis == "x":
            obj["x"] = -obj["x"]
        else:
            obj["y"] = -obj["y"]
    return m


def merge(dst, src, dx=0.0, dy=0.0):
    """Copy all atoms/bonds/notes of src into dst with an offset; returns index map."""
    base = len(dst.atoms)
    for a in src.atoms:
        b = dict(a); b["x"] += dx; b["y"] += dy
        dst.atoms.append(b)
    for b in src.bonds:
        c = dict(b); c["i"] += base; c["j"] += base
        dst.bonds.append(c)
    for n in src.notes:
        c = dict(n); c["x"] += dx; c["y"] += dy
        dst.notes.append(c)
    return base


# ---------------------------------------------------------------------------
# Reaction / pathway scheme: molecules in a row (wrapping), arrows with labels
# ---------------------------------------------------------------------------
def scheme(items, title=None, per_row=3, unit=UNIT_IN, gap=3.2, caption=None, show_now=True):
    """items: [Mol, ("enzyme", "cofactor note"), Mol, (...), Mol ...]
    Molecules are drawn at one scale; arrows carry an enzyme label above and
    an optional cofactor/regulation note below. Rows wrap after per_row molecules."""
    mols = [x for x in items if isinstance(x, Mol)]
    arrows = [x for x in items if not isinstance(x, Mol)]
    exts = [extent(mm, pad=0.4) for mm in mols]
    rows = [list(range(k, min(k + per_row, len(mols)))) for k in range(0, len(mols), per_row)]
    row_h = [max(exts[k][3] - exts[k][2] for k in r) for r in rows]
    row_w = [sum(exts[k][1] - exts[k][0] for k in r) + gap * (len(r) - 1) for r in rows]
    W = max(row_w) + (1.6 if len(rows) > 1 else 0)
    H = sum(row_h) + 1.6 * (len(rows) - 1)
    th = 0.45 if title else 0.0
    ch = 0.4 if caption else 0.0
    fig = plt.figure(figsize=(W * unit, H * unit + th + ch))
    ax = fig.add_axes([0, ch / (H * unit + th + ch), 1, H * unit / (H * unit + th + ch)])
    ax.set_xlim(0, W); ax.set_ylim(-H, 0); ax.set_aspect("equal"); ax.axis("off")
    y_top = 0.0
    pos = {}
    for r, ids in enumerate(rows):
        x = 0.0 if r == 0 else 1.6
        cy = y_top - row_h[r] / 2
        for k in ids:
            x0, x1, y0, y1 = exts[k]
            dx = x - x0; dy = cy - (y0 + y1) / 2
            mm = Mol(); merge(mm, mols[k], dx, dy)
            draw(ax, mm)
            pos[k] = (x, x + (x1 - x0), cy, y1 - y0)
            x += (x1 - x0) + gap
        y_top -= row_h[r] + 1.6
    for k, lab in enumerate(arrows):
        if k + 1 >= len(mols):
            break
        enz, note = (lab, "") if isinstance(lab, str) else (lab[0], lab[1] if len(lab) > 1 else "")
        xa0, xa1, ya, ha = pos[k]; xb0, xb1, yb, hb = pos[k + 1]
        same = abs(ya - yb) < 1e-6
        if same:
            p, q = (xa1 + 0.25, ya), (xb0 - 0.25, yb)
        else:  # wrap: from under the last molecule of a row to above the first of the next
            p, q = ((xa0 + xa1) / 2, ya - ha / 2 + 0.1), ((xb0 + xb1) / 2, yb + hb / 2 - 0.1)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.4, color=COL["ink"],
                                     connectionstyle="arc3,rad=0" if same else "arc3,rad=-0.12"))
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        box = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        if same:
            ax.text(mx, my + 0.3, enz, ha="center", va="bottom", fontsize=8.5, color=COL["ester"],
                    fontweight="bold", bbox=box)
            if note:
                ax.text(mx, my - 0.3, note, ha="center", va="top", fontsize=8, color=COL["muted"], bbox=box)
        else:
            ax.text(mx, my, enz + ("\n" + note if note else ""), ha="center", va="center", fontsize=8.5,
                    color=COL["ester"], bbox=box)
    if title:
        fig.text(0.0, 1 - (th / 2) / (H * unit + th + ch), title, fontsize=11.5, fontweight="bold", va="center")
    if caption:
        fig.text(0.0, (ch / 2) / (H * unit + th + ch), caption, fontsize=9, color=COL["muted"], va="center")
    if show_now:
        plt.show()
    return fig


try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(s): return s


def _md(lines):
    display(Markdown("\n".join(lines) if isinstance(lines, (list, tuple)) else lines))


# ---------------------------------------------------------------------------
# L23 lecture-notes figures and calculators: conformational analysis I
# Butane torsion profile V(phi) = sum_n Vn/2 [1 + cos(n phi - gamma_n)] with
# ILLUSTRATIVE parameters chosen to reproduce textbook butane energies
# (gauche ~3.8, CH3/H eclipsed ~15, syn ~21 kJ/mol); not from one force field.
# ---------------------------------------------------------------------------
import math as _m
import numpy as _np

TORS = [(1, 6.5, 0.0), (2, -1.5, 180.0), (3, 14.5, 0.0)]   # (n, Vn kJ/mol, gamma deg)
R_GAS = 8.314e-3                                           # kJ/mol/K
_BLUE, _RED, _GREY, _GREEN, _ORANGE = "#1d6fb8", "#be123c", "#6b7280", "#15803d", "#c2410c"


def _clean(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25)


def V_torsion(phi, terms=None):
    terms = terms or TORS
    phi = _np.asarray(phi, dtype=float)
    return sum(V / 2 * (1 + _np.cos(_np.radians(n * phi - g))) for n, V, g in terms)


# ---- Newman projections ---------------------------------------------------
def _newman(ax, phi, title):
    ax.set_aspect("equal"); ax.axis("off")
    ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, lw=1.8, color=COL["ink"]))
    ecl = phi % 120 == 0
    off = 12 if ecl else 0                       # nudge eclipsed back bonds so they show
    for k, ang in enumerate((90, 210, 330)):    # front carbon: bonds meet at the centre
        x, y = 1.55 * _m.cos(_m.radians(ang)), 1.55 * _m.sin(_m.radians(ang))
        ax.plot([0, x], [0, y], color=COL["ink"], lw=1.8, zorder=3)
        lab, col = ("CH₃", _RED) if k == 0 else ("H", COL["ink"])
        la = _m.radians(ang - (14 if ecl else 0))
        ax.text(1.95 * _m.cos(la), 1.95 * _m.sin(la), lab, ha="center", va="center",
                fontsize=11, color=col, fontweight="bold" if k == 0 else "normal")
    for k, d in enumerate((0, 120, 240)):       # back carbon: bonds start at the circle
        ang = 90 - phi + d + off
        c, s = _m.cos(_m.radians(ang)), _m.sin(_m.radians(ang))
        ax.plot([c, 1.7 * c], [s, 1.7 * s], color=COL["ink"], lw=1.8, zorder=2)
        lab, col = ("CH₃", _BLUE) if k == 0 else ("H", COL["ink"])
        lb = _m.radians(ang + (14 if ecl else 0))
        ax.text(2.15 * _m.cos(lb), 2.15 * _m.sin(lb), lab, ha="center", va="center", fontsize=11, color=col,
                fontweight="bold" if k == 0 else "normal")
    ax.set_xlim(-2.8, 2.8); ax.set_ylim(-2.6, 2.6)
    ax.set_title(title, fontsize=10.5, fontweight="bold", loc="center")


def fig_butane_newman():
    f, axs = plt.subplots(1, 4, figsize=(12.4, 3.6))
    for ax, (phi, name) in zip(axs, ((0, "syn (fully eclipsed)"), (60, "gauche (staggered)"),
                                     (120, "eclipsed (CH₃ / H)"), (180, "anti (staggered)"))):
        _newman(ax, phi, "%s\nφ = %d°, V = %.1f kJ/mol" % (name, phi, float(V_torsion(phi))))
    plt.tight_layout(); plt.show()
    _md(["Newman projections along C2–C3 of butane. Red CH₃ is on the front carbon (C2), blue CH₃ on the back carbon (C3); "
         "φ is the C1–C2–C3–C4 dihedral angle. Energies come from the illustrative V(φ) in the next figure."])


def fig_butane_profile():
    phi = _np.linspace(0, 360, 721)
    E = V_torsion(phi)
    f, (ax, ax2) = plt.subplots(1, 2, figsize=(13.0, 4.6), gridspec_kw=dict(width_ratios=[2.3, 1]))
    ax.plot(phi, E, color=_BLUE, lw=2.4)
    rows = []
    for k in range(1, len(phi) - 1):
        if (E[k] < E[k - 1] and E[k] <= E[k + 1]) or (E[k] > E[k - 1] and E[k] >= E[k + 1]):
            rows.append((phi[k], E[k], E[k] < E[k - 1]))
    rows = [(0.0, float(E[0]), False)] + rows + [(360.0, float(E[-1]), False)]
    for p, e, ismin in rows:
        if abs(p - 180) < 1:
            nm, col = "anti (global min)", _GREEN
        elif p < 1 or p > 359:
            nm, col = "syn", _RED
        elif ismin:
            nm, col = "gauche (local min)", _GREEN
        else:
            nm, col = "eclipsed", _RED
        ax.plot([p], [e], "o", color=col, ms=7, zorder=5)
        ax.annotate("%s\n%.0f°, %.1f kJ/mol" % (nm, p, e), (p, e), xytext=(0, 9 if not ismin else -30),
                    textcoords="offset points", ha="center", fontsize=8.3, color=col)
    ax.set_xticks(range(0, 361, 60)); ax.set_xlim(-15, 375); ax.set_ylim(-6, 27)
    ax.set_xlabel("C1–C2–C3–C4 dihedral angle φ (degrees; 300° = −60°)")
    ax.set_ylabel("torsional energy (kJ/mol)")
    ax.set_title("A. Butane: V(φ) = Σ Vn/2 [1 + cos(nφ − γn)]", fontsize=10.5, loc="left")
    styles = {1: ":", 2: "-.", 3: "--"}
    for n, V, g in TORS:
        ax2.plot(phi, V / 2 * (1 + _np.cos(_np.radians(n * phi - g))), styles[n], lw=1.6,
                 label="n = %d, V%d = %+.1f, γ = %g°" % (n, n, V, g))
    ax2.plot(phi, E, color=_BLUE, lw=1.0, alpha=0.5, label="sum")
    ax2.set_xticks(range(0, 361, 90)); ax2.set_xlabel("φ (degrees)")
    ax2.legend(fontsize=7.8, frameon=False, loc="upper center", ncol=1)
    ax2.set_ylim(-3, 30)
    ax2.set_title("B. The three cosine terms (kJ/mol)", fontsize=10.5, loc="left")
    for a_ in (ax, ax2):
        _clean(a_)
    plt.tight_layout(); plt.show()


def torsion_energy(phi=60.0, V1=6.5, V2=-1.5, V3=14.5):
    terms = [(1, V1, 0.0), (2, V2, 180.0), (3, V3, 0.0)]
    lines, tot = [], 0.0
    for n, V, g in terms:
        arg = n * phi - g
        e = V / 2 * (1 + _m.cos(_m.radians(arg)))
        tot += e
        lines.append("| n = %d | %g/2 × [1 + cos(%d × %g° − %g°)] = %g/2 × [1 + cos %g°] = %g/2 × [1 + (%.3f)] | **%.2f** |"
                     % (n, V, n, phi, g, V, arg, V, _m.cos(_m.radians(arg)), e))
    _md(["| Term | Working | kJ/mol |", "|---|---|---|"] + lines + ["| **Total V(φ)** | | **%.2f** |" % tot, "",
         "γ = 0° for n = 1 and 3 and 180° for n = 2 (the convention used in the figure). Relative to anti (φ = 180°) "
         "this is %.2f kJ/mol." % (tot - float(V_torsion(180, terms)))])
    return dict(phi=phi, energy=tot, relative_to_anti=tot - float(V_torsion(180, terms)))


# ---- Boltzmann populations --------------------------------------------------
def boltzmann(energies="0, 3.8, 3.8", names="anti, gauche+, gauche−", T=298.15):
    E = [float(x) for x in str(energies).split(",") if x.strip()]
    N = [x.strip() for x in str(names).split(",")]
    N = (N + ["conformer %d" % (k + 1) for k in range(len(N), len(E))])[:len(E)]
    RT = R_GAS * T
    w = [_m.exp(-(e - min(E)) / RT) for e in E]
    Z = sum(w)
    rows = ["| Conformer | ΔE (kJ/mol) | ΔE/RT | weight = e^(−ΔE/RT) | population = weight / Σ |", "|---|---|---|---|---|"]
    for n_, e, wi in zip(N, E, w):
        rows.append("| %s | %.2f | %.3f | %.4f | **%.1f%%** |" % (n_, e - min(E), (e - min(E)) / RT, wi, 100 * wi / Z))
    _md(["1. RT = 0.008314 kJ/mol/K × %.2f K = **%.4f kJ/mol**" % (T, RT),
         "2. Weight of each conformer relative to the lowest: e^(−ΔE/RT)",
         "3. Sum of weights Σ = **%.4f**; each population = weight / Σ" % Z, ""] + rows +
        ["", "Enter a conformer once for each equivalent copy (gauche+ and gauche− are two copies of the same energy)."])
    return {n_: 100 * wi / Z for n_, wi in zip(N, w)}


def heating_factor(barrier=15.0, T_low=300.0, T_high=900.0):
    f1 = _m.exp(-barrier / (R_GAS * T_low)); f2 = _m.exp(-barrier / (R_GAS * T_high))
    _md(["The fraction of molecules with at least the barrier energy Ea is proportional to the Boltzmann factor e^(−Ea/RT):",
         "1. At %.0f K: RT = %.3f kJ/mol; e^(−%.1f/%.3f) = **%.4g**" % (T_low, R_GAS * T_low, barrier, R_GAS * T_low, f1),
         "2. At %.0f K: RT = %.3f kJ/mol; e^(−%.1f/%.3f) = **%.4g**" % (T_high, R_GAS * T_high, barrier, R_GAS * T_high, f2),
         "3. Ratio = %.4g / %.4g = **%.0f-fold** more barrier crossings at the higher temperature" % (f2, f1, f2 / f1),
         "", "This is why molecular dynamics runs used for conformational searching are often heated (for example to ~900 K): "
             "barriers are crossed in a short simulation. Snapshots are then **minimized** to find the nearby low-energy conformations."])
    return dict(factor_low=f1, factor_high=f2, ratio=f2 / f1)


# ---- simple dynamics on the butane torsion ---------------------------------
def _sd_traj(T, n=4000, seed=3):
    """Overdamped Langevin (Brownian) dynamics on V(phi): drift -mu dV/dphi, noise sqrt(2 mu RT)."""
    rng = _np.random.default_rng(seed)
    mu = 4.0                      # mobility x time step (deg^2 per kJ/mol), illustrative
    x = 180.0; out = [x]
    for _ in range(n):
        g = float(V_torsion(x + 0.5) - V_torsion(x - 0.5))
        x = (x - mu * g + _m.sqrt(2 * mu * R_GAS * T) * rng.normal()) % 360.0
        out.append(x)
    return _np.array(out)


def _quench(x):
    for _ in range(400):
        g = float((V_torsion(x + 0.5) - V_torsion(x - 0.5)) / 1.0)
        x = (x - 0.5 * g) % 360.0
    return round(x)


def fig_dynamics():
    f, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.0), gridspec_kw=dict(width_ratios=[2.2, 1]))
    found = {}
    for T, col in ((900, _ORANGE), (300, _GREY)):
        tr = _sd_traj(T)
        a1.plot(_np.arange(len(tr)), tr, ".", ms=1.4, color=col, label="%d K" % T)
        snaps = tr[::200]
        mins = [_quench(s) for s in snaps]
        names = ["gauche" if abs(m_ - 180) > 30 else "anti" for m_ in mins]
        found[T] = (names.count("anti"), names.count("gauche"))
    for y, t_ in ((60, "gauche"), (180, "anti"), (300, "gauche")):
        a1.axhline(y, color=_GREEN, lw=0.8, ls=":")
        a1.text(len(tr) * 1.01, y, t_, va="center", fontsize=8.5, color=_GREEN)
    a1.set_yticks(range(0, 361, 60)); a1.set_ylim(0, 360)
    a1.set_xlabel("simulation step"); a1.set_ylabel("dihedral φ (degrees)")
    a1.legend(markerscale=8, fontsize=8.5, loc="upper left", frameon=True)
    a1.set_title("A. One torsion followed over time at two temperatures", fontsize=10.5, loc="left")
    xs = _np.arange(2)
    for k, (T, col) in enumerate(((300, _GREY), (900, _ORANGE))):
        a2.bar(xs + (k - 0.5) * 0.36, found[T], width=0.34, color=col, label="%d K" % T)
    a2.set_xticks(xs); a2.set_xticklabels(["anti", "gauche (±)"])
    a2.set_ylabel("minimized snapshots (of %d)" % (len(_sd_traj(300)[::200])))
    a2.legend(fontsize=8.5, frameon=False)
    a2.set_title("B. Snapshots minimized: which wells?", fontsize=10.5, loc="left")
    for ax in (a1, a2):
        _clean(ax)
    plt.tight_layout(); plt.show()
    _md(["A toy model of the butane torsion (Brownian dynamics: the force −dV/dφ plus random thermal kicks whose size grows with T; "
         "arbitrary time units, same random numbers at both temperatures). At 300 K the torsion seldom leaves the well it starts in; "
         "at 900 K it crosses the barriers again and again. "
         "Minimizing snapshots taken along the hot trajectory recovers **both** kinds of minimum. Real MD integrates Newton's equations "
         "for every atom with femtosecond steps, but the logic (heat, sample, minimize) is the same."])


# ---- cyclohexane ------------------------------------------------------------
def _chair_xy(h=0.25, r=0.866, tilt=-16.0, spin=-5.0):
    """Project a regular chair (bond length 1) onto the page: ring turned by `spin` about
    its axis and tipped by `tilt` toward the viewer, so all six atoms have distinct x (seen from slightly below the mean plane, the usual textbook view)."""
    s, c = _m.sin(_m.radians(tilt)), _m.cos(_m.radians(tilt))
    P = []
    for k in range(6):
        a = _m.radians(60 * k + spin); z = h if k % 2 == 0 else -h
        x, y = r * _m.cos(a), r * _m.sin(a)
        P.append((x, z * c + y * s))
    return P, (s, c)


def _subst_dir(k, kind, sc, spin=-5.0, beta=19.5):
    s, c = sc
    up = 1 if k % 2 == 0 else -1
    if kind == "ax":
        return (0.0, up * c)
    a = _m.radians(60 * k + spin)
    cb, sb = _m.cos(_m.radians(beta)), _m.sin(_m.radians(beta))
    x, y, z = _m.cos(a) * cb, _m.sin(a) * cb, -up * sb
    return (x, z * c + y * s)


def _chair(ax, x0, subs, title, hl=(), scale=1.9):
    P, sc = _chair_xy()
    P = [(x0 + scale * x, scale * y) for x, y in P]
    for k in range(6):
        (x1, y1), (x2, y2) = P[k], P[(k + 1) % 6]
        ax.plot([x1, x2], [y1, y2], color=COL["ink"], lw=2.2, solid_capstyle="round", zorder=3)
    ends = {}
    for (k, kind, lab, col) in subs:
        dx, dy = _subst_dir(k, kind, sc)
        L = 1.0
        x, y = P[k]
        ax.plot([x, x + L * dx], [y, y + L * dy], color=col, lw=1.9, zorder=2)
        ends[(k, kind)] = (x + L * dx, y + L * dy)
        if lab:
            ax.text(x + (L + 0.3) * dx, y + (L + 0.26) * dy, lab, ha="center", va="center", fontsize=10.5, color=col,
                    fontweight="bold" if lab != "H" else "normal")
    for (i, j) in hl:
        (x1, y1), (x2, y2) = ends[(i, "ax")], ends[(j, "ax")]
        ax.plot([x1, x2], [y1, y2], ls=":", color=_RED, lw=1.4)
    ax.text(x0, 2.7, title, ha="center", fontsize=10.5, fontweight="bold")


def fig_cyclohexane_chair():
    f, ax = plt.subplots(figsize=(13.0, 4.6))
    allsub = []
    for k in range(6):
        allsub.append((k, "ax", "", _RED))
        allsub.append((k, "eq", "", _BLUE))
    _chair(ax, 0.0, allsub, "A. Chair: axial (red) and equatorial (blue)")
    _chair(ax, 4.8, [(0, "ax", "CH₃", _RED), (0, "eq", "H", COL["ink"]), (2, "ax", "H", _ORANGE), (4, "ax", "H", _ORANGE)],
           "B. Axial CH₃: 1,3-diaxial contacts", hl=((0, 2), (0, 4)))
    _chair(ax, 9.6, [(0, "eq", "CH₃", _BLUE), (0, "ax", "H", COL["ink"]), (2, "ax", "H", COL["ink"]), (4, "ax", "H", COL["ink"])],
           "C. Equatorial CH₃: preferred (~95%)")
    ax.set_xlim(-2.6, 12.3); ax.set_ylim(-2.4, 3.0); ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout(); plt.show()
    _md(["Axial bonds are parallel to the ring's axis (alternately up and down); equatorial bonds point outward, roughly "
         "parallel to two ring bonds. A ring flip turns every axial position into equatorial and vice versa. In B the axial CH₃ "
         "is close to the axial H atoms on C3 and C5 (dotted), the same interaction as two gauche-butane contacts."])


def fig_ring_energies():
    pts = [("chair", 0.0), ("half-chair", 45.0), ("twist-boat", 23.0), ("boat", 29.0), ("twist-boat", 23.0),
           ("half-chair", 45.0), ("chair\n(flipped)", 0.0)]
    f, ax = plt.subplots(figsize=(10.5, 3.9))
    for k, (nm, e) in enumerate(pts):
        col = _GREEN if e == 0 else (_RED if nm.startswith("half") else _BLUE)
        ax.plot([k - 0.28, k + 0.28], [e, e], color=col, lw=3.2)
        ax.text(k, e + 2.2, "%s\n%.0f" % (nm, e), ha="center", fontsize=8.8, color=col)
        if k:
            ax.plot([k - 1 + 0.28, k - 0.28], [pts[k - 1][1], e], ls=":", color=_GREY, lw=1)
    ax.set_xticks([]); ax.set_xlim(-0.6, len(pts) - 0.4); ax.set_ylim(-5, 62)
    ax.set_ylabel("energy above chair (kJ/mol)")
    ax.set_title("Cyclohexane ring flip: approximate relative energies (chair → chair)", fontsize=10.5, loc="left")
    _clean(ax); ax.grid(False)
    plt.tight_layout(); plt.show()


def equatorial_fraction(dG=7.3, T=298.15):
    RT = R_GAS * T
    K = _m.exp(dG / RT)
    pe = 100 * K / (1 + K)
    _md(["ΔG (axial − equatorial) = %.2f kJ/mol; T = %.2f K" % (dG, T),
         "1. RT = 0.008314 × %.2f = **%.4f kJ/mol**" % (T, RT),
         "2. K = [equatorial]/[axial] = e^(ΔG/RT) = e^(%.2f/%.4f) = e^%.3f = **%.4g**" % (dG, RT, dG / RT, K),
         "3. %% equatorial = 100 × K/(1 + K) = **%.2f%%**; %% axial = **%.2f%%**" % (pe, 100 - pe)])
    return dict(K=K, percent_equatorial=pe)


# ---- search size --------------------------------------------------------------
def conformer_count(rotatable_bonds=4, increment=30, seconds_each=0.05):
    per = int(round(360.0 / increment))
    n = per ** rotatable_bonds
    sec = n * seconds_each
    units = [("seconds", 1), ("minutes", 60), ("hours", 3600), ("days", 86400), ("years", 86400 * 365)]
    best = [u for u in units if sec / u[1] >= 1] or [units[0]]
    nm, div = best[-1]
    _md(["1. Settings per bond = 360° / %g° = **%d**" % (increment, per),
         "2. Conformations = %d^%d = **%s**" % (per, rotatable_bonds, format(n, ",")),
         "3. Time = %s × %g s = %s s = **%.3g %s**" % (format(n, ","), seconds_each, format(round(sec), ","), sec / div, nm),
         "", "Each extra rotatable bond multiplies the count by %d; the count grows as a power, not a sum." % per])
    return dict(per_bond=per, conformations=n, seconds=sec)


def fig_search_growth(seconds_each=0.05):
    nb = _np.arange(1, 15)
    f, ax = plt.subplots(figsize=(10.5, 4.6))
    for inc, col in ((120, _GREEN), (60, _BLUE), (30, _ORANGE), (10, _RED)):
        ax.plot(nb, (360 / inc) ** nb.astype(float), "-o", ms=4, color=col, label="%d° steps (%d per bond)" % (inc, 360 // inc))
    for sec, t_ in ((3600, "1 hour"), (86400, "1 day"), (86400 * 365, "1 year"), (86400 * 365 * 1000, "1,000 years")):
        ax.axhline(sec / seconds_each, color=_GREY, ls=":", lw=1)
        ax.text(14.3, sec / seconds_each, t_, va="center", fontsize=8.5, color=_GREY)
    ax.set_yscale("log"); ax.set_xticks(nb); ax.set_xlim(0.5, 15.6)
    ax.set_xlabel("number of rotatable bonds"); ax.set_ylabel("conformations to evaluate")
    ax.legend(fontsize=8.5, frameon=False, loc="upper left")
    ax.set_title("Systematic search: (360/increment)^n conformations; dotted lines = time at %.2f s each" % seconds_each,
                 fontsize=10.5, loc="left")
    _clean(ax)
    plt.tight_layout(); plt.show()


# ---- molecules and rotatable bonds -------------------------------------------
def tacrine():
    m = Mol("tacrine")
    M = ring(m, 0, 0, 6, rot=90, labels=["", "", "", "N", "", ""])
    L = fuse(m, M[1], M[2], 6)
    fuse(m, M[5], M[4], 6)
    set_double(m, [(M[0], M[5]), (M[1], M[2]), (M[3], M[4]), (L[2], L[3]), (L[4], L[5])])
    sub(m, M[0], "NH₂", angle=90)
    return m


def rivastigmine():
    m = Mol("rivastigmine")
    B = ring(m, 0, 0, 6, rot=90); kekule(m, B, 0)
    o = sub(m, B[0], "O", angle=90)
    c = sub(m, o, "", angle=30)
    sub(m, c, "O", angle=90, order=2)
    n = sub(m, c, "N", angle=-30)
    sub(m, n, "CH₃", angle=-90)
    e1 = sub(m, n, "", angle=30); sub(m, e1, "CH₃", angle=-30)
    ch = sub(m, B[4], "", angle=-30)
    sub(m, ch, "CH₃", angle=-90)
    na = sub(m, ch, "N", angle=30)
    sub(m, na, "CH₃", angle=30); sub(m, na, "CH₃", angle=-90)
    return m


def lisinopril():
    m = Mol("lisinopril")
    R = 1 / (2 * _m.sin(_m.pi / 5))
    P = ring(m, R, 0, 5, rot=180, labels=["N", "", "", "", ""])
    c = sub(m, P[1], "", angle=270); sub(m, c, "O", angle=210, order=2); sub(m, c, "OH", angle=330)
    ca = sub(m, P[0], "", angle=150); sub(m, ca, "O", angle=90, order=2)
    a = sub(m, ca, "", angle=210)
    ch = grow(m, a, 4, angle=270, first=+1)
    sub(m, ch[-1], "NH₂", angle=300)
    nh = sub(m, a, "NH", angle=150)
    cb = sub(m, nh, "", angle=210)
    co = sub(m, cb, "", angle=270); sub(m, co, "O", angle=330, order=2); sub(m, co, "OH", angle=210)
    c1 = sub(m, cb, "", angle=150); c2 = sub(m, c1, "", angle=210)
    x, y = m.pos(c2)
    ix, iy = x + _m.cos(_m.radians(150)), y + _m.sin(_m.radians(150))
    Ph = ring(m, ix + _m.cos(_m.radians(150)), iy + _m.sin(_m.radians(150)), 6, rot=330)
    kekule(m, Ph, 0); m.bond(c2, Ph[0])
    return m


def count_rotatable_bonds(m, mark=True):
    """Single, non-ring bonds between two non-terminal heavy atoms, excluding the
    C(=O)-N bond of amides and carbamates (partial double-bond character)."""
    deg = [0] * len(m.atoms)
    for b in m.bonds:
        deg[b["i"]] += 1; deg[b["j"]] += 1
    ringb = set()
    for r in getattr(m, "rings", []):
        for k in range(len(r)):
            ringb.add(frozenset((r[k], r[(k + 1) % len(r)])))
    carbonyl = set()
    for b in m.bonds:
        if b["order"] == 2:
            for i, j in ((b["i"], b["j"]), (b["j"], b["i"])):
                if m.atoms[j]["label"] == "O" and m.atoms[i]["label"] == "":
                    carbonyl.add(i)
    n = 0
    for b in m.bonds:
        i, j = b["i"], b["j"]
        if b["order"] != 1 or frozenset((i, j)) in ringb or deg[i] < 2 or deg[j] < 2:
            continue
        li, lj = m.atoms[i]["label"], m.atoms[j]["label"]
        if (i in carbonyl and lj.startswith("N")) or (j in carbonyl and li.startswith("N")):
            continue
        n += 1
        if mark:
            b["hl"] = _RED
    return n


def fig_rotatable():
    ms = [tacrine(), rivastigmine(), lisinopril()]
    ns = [count_rotatable_bonds(m) for m in ms]
    grid(ms, ["A. Tacrine: %d rotatable bonds" % ns[0], "B. Rivastigmine: %d rotatable bonds" % ns[1],
              "C. Lisinopril: %d rotatable bonds" % ns[2]], ncols=3, align="center")
    _md(["Rotatable bonds are shaded red. Rule used: a single bond, not in a ring, between two atoms that each carry another "
         "heavy atom; amide and carbamate C(=O)–N bonds are not counted (partial double bond). Bonds to CH₃, OH or NH₂ are not "
         "counted because turning them moves only hydrogens. Programs differ slightly in these rules, so counts can differ by one or two."])
    return dict(zip(("tacrine", "rivastigmine", "lisinopril"), ns))


def rmsd_calc(deviations="0.2, 0.4, 0.3, 0.5"):
    d = [float(x) for x in str(deviations).split(",") if x.strip()]
    sq = [x * x for x in d]
    mean = sum(sq) / len(sq)
    r = _m.sqrt(mean)
    _md(["1. Square each matched-atom distance: " + ", ".join("%.3g² = %.4g" % (x, s) for x, s in zip(d, sq)),
         "2. Mean of the squares = %.4g / %d = **%.4g Å²**" % (sum(sq), len(sq), mean),
         "3. RMSD = √%.4g = **%.3f Å**" % (mean, r)])
    return dict(rmsd=r)

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
# ===== END kit =====


## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Differentiate a local from a global energy minimum on an energy diagram. | Understand | V | 6 |
| 2 | Construct a steric energy versus dihedral angle profile and identify the minima. | Apply | IV | 6 |
| 3 | Calculate how many conformations a stepwise search must evaluate for n rotatable bonds. | Apply | IV | 6 |
| 4 | Recommend a search method for a rigid, cyclic or highly flexible ligand. | Analyze | III | 3 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  A minimization run on one starting geometry returns a conformation. What has it established?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. That this is the lowest-energy shape the molecule can take", "B. That every small change from this shape raises the energy", "C. That this is the shape the molecule takes in the binding site", "D. That no other conformation of the molecule is accessible"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  Why does a systematic search over rotatable bonds become impossible for a flexible drug?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Because each additional bond adds its increments to the total", "B. Because the force field loses accuracy as bond count increases", "C. Because each additional bond multiplies the number of structures", "D. Because rotatable bonds interact and cannot be scanned at all"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  In molecular dynamics, what is the purpose of raising the simulated temperature?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. To supply kinetic energy so barriers between minima are crossed", "B. To lower the barriers themselves so less energy is required", "C. To remove the non-bonded terms that hold the molecule folded", "D. To speed the calculation by taking fewer integration steps"]
gb.answer("P3", response[:1])

## Part 2 - Key ideas: conformations, minima and the search problem


**Summary.** Rotating about single bonds changes a molecule's shape without breaking any bond; each shape is a **conformation**. A flexible drug can adopt thousands of them, and the one that binds the target (the **bioactive conformation**) is not necessarily the lowest in energy. So before a 3D model can be used for design or docking, we need to know which conformations are possible and how much each costs.

Energy minimization (L22) cannot answer that on its own: it goes downhill from wherever it starts and stops in the **nearest** well, a **local minimum**. The lowest well of all is the **global minimum**. To find more than one minimum we must *search* the energy surface:

| Search method | How new structures are made | Strength | Weakness | Best for |
|---|---|---|---|---|
| **Systematic (stepwise / grid)** | rotate every rotatable bond through fixed increments; score (and usually minimize) every combination | exhaustive at its resolution; reproducible | count = (360/increment)^n, explodes with n; coarse grids miss narrow wells | rigid or nearly rigid ligands (few rotatable bonds) |
| **Molecular dynamics (MD)** | Newton's equations of motion in femtosecond steps, often at high temperature; snapshots are minimized | handles rings and coupled motions; samples real motion | needs long runs; heating can distort; no guarantee of the global minimum | cyclic molecules, proteins, flexible complexes |
| **Stochastic (Monte Carlo, simulated annealing, genetic algorithms)** — L24 | random changes, accepted or rejected by an energy rule | copes with many rotatable bonds | random, so must be repeated to be confident | highly flexible ligands |

A **rotatable bond** is a single, non-ring bond whose rotation moves heavy atoms. Amide C–N bonds are normally excluded (partial double bond), and rings are treated as units because their bonds cannot rotate freely.

<details>
<summary><b>▸ Key terms (open)</b></summary>


| Term | Meaning |
|---|---|
| Conformation | One 3D arrangement reached by rotation about single bonds (no bonds broken) |
| Configuration | Arrangement that can only change by breaking bonds (for example R vs S) |
| Dihedral (torsion) angle φ | Angle between the planes A–B–C and B–C–D for four bonded atoms A–B–C–D |
| Staggered / eclipsed | Substituents on adjacent carbons as far apart as possible (φ = 60°, 180°) / lined up (0°, 120°) |
| Anti / gauche | Staggered with the two largest groups at 180° / at ±60° |
| Local minimum | A conformation from which every small change raises the energy |
| Global minimum | The lowest-energy local minimum |
| Bioactive (active) conformation | The conformation adopted in the binding site |
| Systematic (grid) search | Rotating every rotatable bond through fixed increments |
| Molecular dynamics | Following atoms over time with Newton's laws and a force field |
| Boltzmann distribution | Population of each conformer ∝ e^(−ΔE/RT) |
| Axial / equatorial | Ring substituent bonds parallel to the ring axis / pointing outward around the ring's equator |
| 1,3-Diaxial interaction | Steric clash between an axial group and the axial H atoms two carbons away |

</details>

**Which search for which ligand?**

| Ligand type | Example | Rotatable bonds (our rule) | Sensible approach |
|---|---|---|---|
| Rigid, fused rings | tacrine (1ACJ) | 0 | none needed beyond minimization; ring pucker only |
| Ring that flips or puckers | cyclohexane, piperidine, sugars | 0 (but the ring itself moves) | molecular dynamics, then minimize snapshots |
| A few rotatable bonds | rivastigmine | 5 | systematic search at 30° is feasible (12⁵ ≈ 250,000) |
| Many rotatable bonds | lisinopril (1O86), indinavir (1HSG) | 12 or more | stochastic search or MD; systematic search is impossible |

## Part 3 - Butane: one torsion, computed


Butane is the smallest molecule with a meaningful torsion profile. Looking down the C2–C3 bond (a **Newman projection**), the two methyl groups can be anti (180°), gauche (±60°), or eclipsed with a hydrogen (±120°) or with each other (syn, 0°).

Force fields describe each torsion with a short **cosine series**:

**V(φ) = Σ (Vₙ/2) [1 + cos(nφ − γₙ)]**, n = 1, 2, 3

The n = 3 term gives the three-fold barrier of any sp³–sp³ bond (as in ethane); the n = 1 term makes the syn end higher than the anti end (methyl–methyl repulsion); n = 2 fine-tunes the shape. Here we use **illustrative** values V₁ = 6.5, V₂ = −1.5, V₃ = 14.5 kJ/mol (γ = 0°, 180°, 0°), chosen so the curve reproduces the textbook butane energies: gauche ≈ 3.8, CH₃/H eclipsed ≈ 15 and syn ≈ 21 kJ/mol above anti.

In [ ]:
#@title 3.1 Figure — Newman projections of butane at four dihedral angles { display-mode: "form" }
fig_butane_newman()

In [ ]:
#@title 3.2 Figure — computed torsion energy profile of butane { display-mode: "form" }
fig_butane_profile()

Reading the profile: **minima** (green) are staggered conformations; **maxima** (red) are eclipsed. Anti is the **global** minimum; the two gauche wells are **local** minima. With this parameter set the gauche minima sit at about 64° and 296° rather than exactly 60°, because the n = 1 term tilts the well; real scans show the same small shift. A minimization started at 100° would roll into the gauche well, not the anti one, which is exactly what you will test in Part 7.

#### Worked example: energy of the gauche conformation by hand
With V₁ = 6.5, V₂ = −1.5, V₃ = 14.5 kJ/mol and γ = 0°, 180°, 0°, calculate V(60°).

<details>
<summary><b>▸ Worked solution</b></summary>

1. n = 1: 6.5/2 × [1 + cos(60°)] = 3.25 × 1.5 = **4.875**
2. n = 2: −1.5/2 × [1 + cos(120° − 180°)] = −0.75 × [1 + cos(−60°)] = −0.75 × 1.5 = **−1.125**
3. n = 3: 14.5/2 × [1 + cos(180°)] = 7.25 × 0 = **0**
4. Sum: 4.875 − 1.125 + 0 = **3.75 kJ/mol** above anti (V(180°) = 0 for this set).

The three-fold term vanishes at every staggered angle, so the gauche–anti difference comes entirely from the n = 1 and n = 2 terms. Check with the calculator, then try φ = 0° and 120°.
</details>

In [ ]:
#@title 3.3 Calculator — torsion energy term by term { display-mode: "form" }
dihedral = 60 #@param {type:"slider", min:0, max:360, step:5}
V1 = 6.5 #@param {type:"number"}
V2 = -1.5 #@param {type:"number"}
V3 = 14.5 #@param {type:"number"}
_ = torsion_energy(dihedral, V1, V2, V3)

## Part 4 - How much of each conformer? Boltzmann populations and temperature


A sample of butane is not all anti. At room temperature molecules interconvert millions of times a second, and the fraction in each well follows the **Boltzmann distribution**:

population of conformer *i* ∝ gᵢ e^(−ΔEᵢ/RT), where RT = 2.48 kJ/mol at 298 K and gᵢ counts equivalent copies (gauche+ and gauche− give g = 2).

Two rules of thumb follow: **ΔE ≈ 5.7 kJ/mol is a 10-fold population difference** at room temperature, and conformers more than about 10 kJ/mol above the minimum are present at only a few percent or less. Strictly the free energy (ΔG) should be used; steric energy differences are an approximation.

#### Worked example: anti versus gauche butane at 298 K
Gauche butane is 3.8 kJ/mol above anti, and there are two gauche conformers. What percentage of butane is anti at 298 K?

<details>
<summary><b>▸ Worked solution</b></summary>

1. RT = 0.008314 × 298 = **2.478 kJ/mol**.
2. Weight of each gauche relative to anti: e^(−3.8/2.478) = e^(−1.533) = **0.216**.
3. Sum of weights: 1 (anti) + 0.216 + 0.216 = **1.432**.
4. Anti = 1/1.432 = **69.8%**; each gauche = 0.216/1.432 = **15.1%** (30% gauche in total).

Even though anti is the global minimum, almost a third of molecules are gauche. Enter `0, 3.8, 3.8` in the calculator to check.
</details>

In [ ]:
#@title 4.1 Calculator — Boltzmann populations, step by step { display-mode: "form" }
relative_energies_kJ = "0, 3.8, 3.8" #@param {type:"string"}
conformer_names = "anti, gauche+, gauche-" #@param {type:"string"}
temperature_K = 298.15 #@param {type:"number"}
_ = boltzmann(relative_energies_kJ, conformer_names, temperature_K)

**Why molecular dynamics is heated.** In MD the atoms move according to Newton's laws under the force field, with time steps of about a femtosecond (10⁻¹⁵ s). At 300 K a simulation of practical length may never cross a 15 kJ/mol barrier, so it only samples the well it started in. Raising the simulated temperature (for example to ~900 K) gives the molecule enough kinetic energy to hop between wells. Snapshots are taken along the trajectory and each is **minimized**; the minimized structures are compared and the lowest is kept. The figure uses a toy model of the butane torsion to show this.

In [ ]:
#@title 4.2 Figure — heating lets a simulation cross barriers; minimizing snapshots finds the wells { display-mode: "form" }
fig_dynamics()

#### Worked example: how much does heating help?
Compare the Boltzmann factor e^(−Ea/RT) for a 15 kJ/mol barrier at 300 K and at 900 K.

<details>
<summary><b>▸ Worked solution</b></summary>

1. 300 K: RT = 0.008314 × 300 = 2.494 kJ/mol; e^(−15/2.494) = e^(−6.01) = **0.0024**.
2. 900 K: RT = 7.483 kJ/mol; e^(−15/7.483) = e^(−2.00) = **0.135**.
3. Ratio: 0.135 / 0.0024 ≈ **55**.

About 55 times more barrier crossings per unit time at 900 K, so the same simulated time samples far more of the energy surface.
</details>

In [ ]:
#@title 4.3 Calculator — barrier crossing and temperature { display-mode: "form" }
barrier_kJ = 15 #@param {type:"slider", min:2, max:60, step:1}
T_low = 300 #@param {type:"number"}
T_high = 900 #@param {type:"number"}
_ = heating_factor(barrier_kJ, T_low, T_high)

## Part 5 - Rings: the cyclohexane chair, axial and equatorial


Ring bonds cannot rotate freely, but rings still change conformation. **Cyclohexane** is puckered: in the **chair** every C–C bond is staggered and every angle is close to tetrahedral, so it has essentially no angle or torsional strain. Each carbon carries one **axial** bond (parallel to the ring axis) and one **equatorial** bond (pointing outward). A **ring flip** passes through higher-energy half-chair, twist-boat and boat forms and turns every axial position into equatorial.

In [ ]:
#@title 5.1 Figure — chair cyclohexane: axial and equatorial positions, methyl axial vs equatorial { display-mode: "form" }
fig_cyclohexane_chair()

In [ ]:
#@title 5.2 Figure — energies along the cyclohexane ring flip { display-mode: "form" }
fig_ring_energies()

| | Axial substituent | Equatorial substituent |
|---|---|---|
| Direction | parallel to the ring axis, alternately up and down | outward, roughly in the ring's mean plane |
| Neighbours it meets | axial H atoms on C3 and C5 (1,3-diaxial) | nothing close; anti to ring carbons |
| Equivalent butane interactions | two gauche-butane contacts | none |
| Preference | disfavoured, more so for bigger groups | favoured |
| After a ring flip | becomes equatorial | becomes axial |

Approximate axial-minus-equatorial free energies (**A-values**) at room temperature, in kJ/mol: F ≈ 1.0; Cl ≈ 2.2; OH ≈ 3.6 (solvent-dependent); CH₃ ≈ 7.3; ethyl ≈ 7.5; isopropyl ≈ 9.2; phenyl ≈ 11.7; tert-butyl ≈ 20 or more (it effectively locks the ring).

#### Worked example: methylcyclohexane, and where 7.3 kJ/mol comes from
(a) Use butane to estimate the energy cost of an axial methyl group. (b) With the measured value ΔG = 7.3 kJ/mol, what percentage of methylcyclohexane has the methyl equatorial at 298 K?

<details>
<summary><b>▸ Worked solution</b></summary>

(a) An axial methyl is gauche to ring carbons C3 and C5: **two** gauche-butane interactions. An equatorial methyl is anti to both. Estimate: 2 × 3.8 = **7.6 kJ/mol**, close to the measured 7.3.

(b) 1. RT = 2.479 kJ/mol. 2. K = [eq]/[ax] = e^(7.3/2.479) = e^(2.945) = **19.0**. 3. % equatorial = 100 × 19.0/20.0 = **95%** (5% axial).

The same reasoning explains why a bulky substituent on a ring in a drug usually sits equatorial, and why a tert-butyl group can hold a ring in one chair.
</details>

In [ ]:
#@title 5.3 Calculator — percent equatorial from the A-value { display-mode: "form" }
A_value_kJ = 7.3 #@param {type:"number"}
temperature_K = 298.15 #@param {type:"number"}
_ = equatorial_fraction(A_value_kJ, temperature_K)

## Part 6 - The size of a systematic search


A systematic search tries every combination of torsion settings. With an increment of θ degrees there are 360/θ settings per bond, and the settings **multiply**:

**number of conformations = (360/θ)ⁿ** for n rotatable bonds; **time = number × time per conformation**.

So the count grows as a power of n. A finer grid is more reliable (narrow wells are not stepped over), but it raises the base of the power. The activity in Part 8 uses 0.05 s per conformation; the same value is used here.

In [ ]:
#@title 6.1 Figure — conformations to evaluate versus rotatable bonds { display-mode: "form" }
fig_search_growth()

#### Worked example: four bonds versus seven bonds
How many conformations does a 30° systematic search generate for 4 and for 7 rotatable bonds, and how long does each take at 0.05 s per conformation? How does a 10° grid change the 4-bond search?

<details>
<summary><b>▸ Worked solution</b></summary>

Settings per bond = 360/30 = 12.

* 4 bonds: 12⁴ = **20,736** conformations × 0.05 s = 1,037 s ≈ **17 minutes**.
* 7 bonds: 12⁷ = **35,831,808** × 0.05 s = 1.79 × 10⁶ s ≈ **21 days**.
* 4 bonds at 10°: 36⁴ = **1,679,616** × 0.05 s = 83,981 s ≈ **23 hours**.

Three more bonds multiplied the work by 12³ = 1,728; a three-fold finer grid multiplied it by 3⁴ = 81. Check each with the calculator.
</details>

In [ ]:
#@title 6.2 Calculator — conformer count and search time { display-mode: "form" }
rotatable_bonds = 4 #@param {type:"slider", min:1, max:15, step:1}
increment_degrees = 30 #@param [10, 15, 30, 60, 120] {type:"raw"}
seconds_per_conformation = 0.05 #@param {type:"number"}
_ = conformer_count(rotatable_bonds, increment_degrees, seconds_per_conformation)

**Counting rotatable bonds.** The figure applies one rule to three drugs used in this course: tacrine (the rigid ligand in 1ACJ), rivastigmine (another acetylcholinesterase inhibitor) and lisinopril (the flexible ACE inhibitor in PDB 1O86).

In [ ]:
#@title 6.3 Figure — rotatable bonds in a rigid, a moderately flexible and a very flexible drug { display-mode: "form" }
fig_rotatable()

#### Worked example: choosing the search from the bond count
Using the counts in the figure, which search would you use for each drug?

<details>
<summary><b>▸ Worked solution</b></summary>

* **Tacrine (0):** no torsional search; minimize and, if needed, check the pucker of the CH₂ ring (a short MD run would do).
* **Rivastigmine (5):** a 30° systematic search is 12⁵ = 248,832 conformations ≈ 3.5 hours at 0.05 s each: feasible and exhaustive.
* **Lisinopril (12):** 12¹² ≈ 8.9 × 10¹² conformations; a systematic search is impossible. Use molecular dynamics or a stochastic method (L24), and use the crystal structure (1O86) to check the answer.
</details>

## Part 7 - Where minimization stops

Below is a relaxed torsion profile for a butane-like fragment: steric energy
against the dihedral angle. The **global minimum** is the anti conformation at
180 degrees; the two **gauche** wells near plus and minus 60 degrees are local
minima.

The marker shows where a downhill minimization **starting from the angle you
choose** comes to rest. Try a start angle on each side of a barrier.

<details>
<summary><b>Stuck? Open this before you ask a neighbour</b></summary>

Minimization only ever goes **downhill**. Put your finger on the starting
angle and slide it down the slope you are standing on — you stop at the
bottom of *that* valley, whatever is on the far side of the next hill.

So the question is never *where is the lowest point?* but *which valley did I
start in?*

</details>

In [ ]:
#@title Torsion profile and where minimization lands { display-mode: "form" }
#@markdown Choose where minimization starts, then press play.
minimisation_starts_at = 100 #@param {type:"slider", min:-180, max:180, step:10}
threefold_barrier_kJ = 14 #@param {type:"slider", min:2, max:30, step:1}
gauche_strain_kJ = 3.8 #@param {type:"slider", min:0, max:12, step:0.2}
r = sims.torsion_profile(threefold_barrier_kJ, gauche_strain_kJ, 10,
                         minimisation_starts_at)
print(f"global minimum at        {r['global_min']:.0f} degrees")
print(f"local minima at          {', '.join(f'{m:.0f}' for m in r['local_minima'])} degrees")
print(f"barrier at 0 degrees     {r['barrier_at_0']:.1f} kJ/mol")
print(f"minimization stopped at  {r['stops_at']:.0f} degrees")

**Now put two settings side by side.** Run the cell below and compare the two figures it prints, one above the other.

In [ ]:
#@title Side by side: two starting angles { display-mode: "form" }
#@markdown Two starting points, same molecule, same force field.
start_A = 100 #@param {type:"slider", min:-180, max:180, step:10}
start_B = 170 #@param {type:"slider", min:-180, max:180, step:10}
print("=" * 70); print(f"SETTING A - minimization starts at {start_A} degrees"); print("=" * 70)
a = sims.torsion_profile(14, 3.8, 10, start_A)
print("=" * 70); print(f"SETTING B - minimization starts at {start_B} degrees"); print("=" * 70)
b = sims.torsion_profile(14, 3.8, 10, start_B)
print()
print(f"A stopped at {a['stops_at']:.0f} degrees, B stopped at {b['stops_at']:.0f} degrees.")
print("Same molecule. Same method. Different answer." if a['stops_at'] != b['stops_at']
      else "Both landed in the same well - try starting angles further apart.")

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  Starting at 100 degrees, minimization stopped at 60 degrees rather than at the global minimum. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The 60 degree well is lower in energy than the 180 degree well", "B. A barrier lies between 100 degrees and the deeper 180 degree well", "C. The algorithm found the global minimum but reported it as 60", "D. The force field has no parameters for angles beyond 90 degrees"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  You raised the gauche strain until the wells near 60 degrees disappeared. What happens to minimization from 100 degrees now?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It stops at 60 degrees, because that minimum is always present", "B. It reaches 180 degrees, because the slope now runs all the way", "C. It stops at 100 degrees, because there is nowhere left to go", "D. It fails, because a profile with one minimum cannot be searched"]
gb.answer("C2", response[:1])

## Part 8 - How big is the search?

If a local minimum depends on where you start, the obvious fix is to try
every starting point. The cell below counts how many structures that takes:
**(360 / increment) raised to the power of the number of rotatable bonds**,
timed at a generous 0.05 seconds each.

Start with 3 bonds. Then try 7. Then try 14, which is an ordinary flexible
drug molecule.

<details>
<summary><b>Stuck? Open this before you ask a neighbour</b></summary>

Do not reach for the slider first — predict. If one bond at 30 degree steps
gives 12 structures, two bonds give 12 x 12. What do seven bonds give?

Then check. The point of this cell is the moment your intuition breaks.

</details>

In [ ]:
#@title Count the conformations { display-mode: "form" }
#@markdown How many structures must a stepwise search evaluate?
rotatable_bonds = 3 #@param {type:"slider", min:1, max:14, step:1}
angle_increment_degrees = 30 #@param [10, 15, 30, 60, 120] {type:"raw"}
r = sims.search_cost(rotatable_bonds, angle_increment_degrees)
print(f"structures to evaluate : {r['conformations']:,}")
print(f"at 0.05 s each         : {r['hours']:,.1f} hours")
print(f"                       : {r['years']:,.3f} years")
if r['years'] > 1:
    print("\nThis job does not finish. A stepwise search is the wrong tool here.")

**Now put two settings side by side.** Run the cell below and compare the two figures it prints, one above the other.

In [ ]:
#@title Side by side: does a coarser grid rescue you? { display-mode: "form" }
#@markdown Same molecule, two grid spacings.
bonds = 10 #@param {type:"slider", min:1, max:14, step:1}
fine_increment = 15 #@param [10, 15, 30, 60, 120] {type:"raw"}
coarse_increment = 120 #@param [10, 15, 30, 60, 120] {type:"raw"}
a = sims.search_cost(bonds, fine_increment)
b = sims.search_cost(bonds, coarse_increment)
print(f"{bonds} rotatable bonds")
print(f"  {fine_increment:>4} degree steps : {a['conformations']:>22,}  ({a['years']:,.2f} years)")
print(f"  {coarse_increment:>4} degree steps : {b['conformations']:>22,}  ({b['years']:,.2f} years)")
print(f"\nCoarsening the grid cut the work by a factor of {a['conformations']/b['conformations']:,.0f}.")
print("But a 120 degree grid can step straight over a narrow minimum and miss it.")

In [ ]:
#@title C3   (run after choosing) { display-mode: "form" }
#@markdown **C3.**  At 30 degree increments, 14 rotatable bonds gave a number beyond any practical computation. What does this establish about stepwise searching?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It is unusable for flexible molecules because cost grows as a power", "B. It is unusable for flexible molecules because the force field fails", "C. It is usable if the computer available is sufficiently powerful", "D. It is usable because most of those structures are chemically alike"]
gb.answer("C3", response[:1])

In [ ]:
#@title C4   (run after choosing) { display-mode: "form" }
#@markdown **C4.**  Coarsening from 15 to 120 degree increments cut the work enormously. What is the cost of that choice?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Narrow energy minima can fall between grid points and be missed", "B. The force field can no longer calculate the steric energy at all", "C. The global minimum is found but its energy is reported wrongly", "D. Only cyclic molecules can be searched once the grid is coarse"]
gb.answer("C4", response[:1])

## Part 9 - Rotamers in a real binding site

The torsion profile above was a model of one bond. A real protein
has thousands. The cell below measures the **chi-1 torsion of every side
chain lining the tacrine site** and classifies each as gauche+, trans or
gauche-.

> **About this structure.** 1ACJ is *Torpedo californica*
> acetylcholinesterase with tacrine bound, from the Protein Data Bank
> (Harel *et al.*, *Proc. Natl. Acad. Sci. USA* 1993, **90**, 9031).
> The coordinates are embedded in this notebook, so the cell below
> downloads nothing and sends nothing anywhere. No personal data of any
> kind is involved, and nothing you type is transmitted or stored.

Before you run it, predict: will the measured values be scattered evenly
around the circle, or clustered?

In [ ]:
#@title Measure side-chain rotamers { display-mode: "form" }
#@markdown How close to the ligand must a residue be to count?
within_angstrom_of_ligand = 8.0 #@param {type:"slider", min:5.0, max:12.0, step:0.5}
r = crystal.rotamers(near_ligand=within_angstrom_of_ligand)

In [ ]:
#@title C5   (run after choosing) { display-mode: "form" }
#@markdown **C5.**  At 8 angstrom your output measures 15 side chains, and every one lies within 40 degrees of a staggered value. What does that clustering establish?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Side chains occupy a few low-energy rotamers rather than arbitrary torsions", "B. Refinement restrained the torsions to ideal values, so this is circular", "C. Fifteen residues is far too small a sample to show any real preference", "D. Binding-site side chains are unusually strained, as the clustering shows"]
gb.answer("C5", response[:1])

In [ ]:
#@title C6   (run after choosing) { display-mode: "form" }
#@markdown **C6.**  Your output assigns Trp84 a chi-1 near -53 degrees and Phe330 near 157 degrees. Both are aromatic residues that stack against the ligand. What follows for a docking calculation that keeps the protein rigid?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Only if it starts from these rotamers, since it cannot find them itself", "B. Yes regardless, because rigid docking still samples the side chains", "C. It fails outright, since rigid docking reproduces no crystal pose at all", "D. Rotamers are irrelevant here, since docking only moves the ligand"]
gb.answer("C6", response[:1])

## Part 10 - How big is the search, really

Now put a number on the combinatorics for this specific molecule in
this specific site, at a resolution you choose.

In [ ]:
#@title Count the search { display-mode: "form" }
#@markdown Rotatable bonds to sample, and how finely.
rotatable_bonds = 7 #@param {type:"slider", min:1, max:14, step:1}
degrees_per_step = 30 #@param [10, 15, 30, 45, 60] {type:"raw"}
r = crystal.search_cost(n_rot=rotatable_bonds, step=degrees_per_step)

In [ ]:
#@title C7   (run after choosing) { display-mode: "form" }
#@markdown **C7.**  Set rotatable_bonds to 12, keeping 30 degree steps: the output reports about 8.9 trillion conformations and some 14,000 years of computing. Halving the step to 15 degrees does what?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Multiplies it by two to the power of twelve, about four thousandfold", "B. Doubles it, since each bond now has twice as many settings to try", "C. Leaves it unchanged, since the number of bonds has not changed", "D. Halves it, since finer steps make the search converge much sooner"]
gb.answer("C7", response[:1])

## Part 11 - Looking ahead: stochastic searches and comparing conformations (L24 preview)


The end-of-activity questions also touch on methods from the next lecture. In brief:

| Method | How a new structure is generated | How it is accepted | Why it helps |
|---|---|---|---|
| **Monte Carlo** | random rotation of one or more single bonds (or random atom moves), then minimization | keep if lower in energy; if higher, keep with probability e^(−ΔE/RT) (Metropolis rule) | escapes local wells without a time-consuming trajectory |
| **Simulated annealing** | Monte Carlo (or MD) repeated at falling temperatures | uphill moves accepted less and less often as T drops | settles into deep minima, like slow cooling of a melt |
| **Genetic algorithm** | each torsion is a "gene"; conformations are "chromosomes" that are mutated and recombined | the fittest (lowest energy or best score) survive to the next generation | searches many bonds at once; used in docking (L25–L26) |

**Comparing two conformations: RMSD.** After the two structures are overlaid (fitted), the root-mean-square deviation is

RMSD = √[ (1/N) Σ dᵢ² ], where dᵢ is the distance between matched atom pair *i*.

It is a distance (Å), not an energy. Below about 1–2 Å two poses or conformations are usually considered the same. A **rigid active analogue** has few conformations, so its shape fixes the geometry of the binding groups; that is why rigid analogues are preferred when defining a 3D pharmacophore.

#### Worked example: calculating an RMSD
Four matched atoms in two overlaid conformations are 0.2, 0.4, 0.3 and 0.5 Å apart. What is the RMSD?

<details>
<summary><b>▸ Worked solution</b></summary>

1. Squares: 0.04, 0.16, 0.09, 0.25 Å².
2. Mean: 0.54 / 4 = **0.135 Å²**.
3. Square root: √0.135 = **0.37 Å**.

Note that the RMSD (0.37 Å) is larger than the simple mean distance (0.35 Å): squaring weights the worst-fitting atoms more.
</details>

In [ ]:
#@title 11.1 Calculator — RMSD from matched-atom distances { display-mode: "form" }
matched_atom_distances = "0.2, 0.4, 0.3, 0.5" #@param {type:"string"}
_ = rmsd_calc(matched_atom_distances)

## Part 12 - Conformation in marketed drugs and their targets (reference)


| Drug (PDB in this course) | Target | Conformational point |
|---|---|---|
| **Tacrine** (1ACJ) | acetylcholinesterase | rigid and nearly planar: its single low-energy shape is the bound shape; only the CH₂ ring puckers. The surrounding side chains sit in staggered rotamers (Part 9) |
| **Lisinopril** (1O86) | angiotensin-converting enzyme | many rotatable bonds; the crystal structure shows the extended conformation that places the carboxylate on Zn²⁺ and the side chains in pockets |
| **Indinavir** (1HSG) | HIV-1 protease | large, flexible peptidomimetic; designed to adopt a conformation matching the enzyme's substrate-binding cleft |
| **Methotrexate** (3DFR) | dihydrofolate reductase | its pteridine ring binds flipped relative to the substrate dihydrofolate, so similar 2D structures can bind in different orientations |
| **Raloxifene** (1ERR) | estrogen receptor α | its side chain holds the receptor's helix 12 out of the active position: a *protein* conformational change explains antagonism |
| **Imatinib** | BCR-ABL kinase | binds and stabilizes an inactive kinase conformation (DFG-out), not the active one |
| **Gefitinib** (4WKQ), **aliskiren** (2V0Z) | EGFR kinase; renin | the bound pose in each crystal structure is the reference a conformational search or docking run should reproduce |

**Looking ahead.** L24 adds stochastic searching, overlaying molecules and pharmacophores; L25–L26 use the same energy terms and search ideas to dock ligands into binding sites.

## Part 13 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 6**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  A conformational search returns a structure the program labels a minimum. What has been established about it?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It is the most stable conformation the molecule can adopt", "B. It is the conformation observed in the crystal structure", "C. It is the conformation the molecule adopts when it binds", "D. It is a stable form, not necessarily the lowest in energy"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  Which feature distinguishes a Monte Carlo search from molecular dynamics?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It follows Newton's equations to produce a time trajectory", "B. It generates structures by random rotation about single bonds", "C. It encodes each torsion as a gene and recombines the strings", "D. It rotates each rotatable bond through fixed equal increments"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  In simulated annealing, why is the Monte Carlo run repeated at progressively lower temperatures?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Acceptance becomes stricter, settling the search in deep minima", "B. Cooling physically strengthens the bonds in the stable conformer", "C. Low temperatures let the program sample more diverse structures", "D. Each cycle removes one rotatable bond from the search problem"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  Two analogues are overlaid and the program reports RMSD = 0.35 A. What does that number describe?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The difference in steric energy between the two conformations", "B. The predicted difference in binding affinity for the two ligands", "C. The average separation of the matched atom pairs after fitting", "D. The proportion of molecular volume shared by the two structures"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  Why is a rigid active analogue the preferred starting point for defining a 3D pharmacophore?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Rigid molecules are usually more potent than flexible ones", "B. Its crystal structure can be solved without a protein present", "C. Rigid structures are easier to synthesize in useful quantity", "D. Its few conformations fix the geometry of the binding groups"]
gb.answer("Q5", response[:1])

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L23|...` line printed above.
2. Paste it into the Canvas assignment **L23 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.